<a href="https://colab.research.google.com/github/angeruzzi/home-credit-default-risk/blob/main/notebooks/03_data_preparation_and_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Home Credit Default Risk: Preparação dos Dados e Feature Engineering

## Introdução

### 1. Contexto

A análise exploratória identificou valores ausentes, códigos especiais, distribuições assimétricas, categorias raras, variáveis correlacionadas e oportunidades de criação de features com significado para risco de crédito.

Este notebook transforma essas evidências em um processo reprodutível de preparação da base de modelagem.

A tabela principal continuará sendo `application_train.csv`, que contém as características observadas no momento da aplicação e o `TARGET`. A tabela `application_test.csv`, disponibilizada sem a variável resposta, não será utilizada na avaliação supervisionada dos modelos.

Para permitir o treinamento, a seleção e a avaliação final dos modelos, `application_train` será dividida em três amostras estratificadas:

- `train`: ajuste dos preprocessadores e treinamento dos modelos;
- `validation`: comparação de alternativas, seleção de features e ajuste de hiperparâmetros;
- `test`: avaliação final após a definição da abordagem.

Como a base não disponibiliza uma data de referência confiável para a aplicação, essa divisão não representa uma validação `Out-of-Time`.

Todas as transformações que aprendem parâmetros dos dados — como imputação, agrupamento de categorias, `binning`, encoding, scaling e seleção de features — deverão ser ajustadas exclusivamente na amostra de treino. Essa separação é necessária para evitar `data leakage`.

### 2. Objetivos

Este notebook possui os seguintes objetivos:

1. carregar e validar a tabela `application_train`;
2. separar identificador, features e `TARGET`;
3. criar as amostras de `train`, `validation` e `test` de forma estratificada;
4. tratar códigos especiais identificados durante a EDA;
5. implementar features temporais e financeiras candidatas;
6. definir regras reprodutíveis para tratamento de valores ausentes e categorias raras;
7. verificar a consistência das transformações entre as amostras;
8. persistir as bases intermediárias para os notebooks de modelagem;
9. registrar metadados, parâmetros e decisões de preparação.

O resultado será uma base analítica adequada para o desenvolvimento de duas abordagens:

- modelo tradicional de credit scoring com `binning`, WoE e Logistic Regression;
- modelo challenger com LightGBM.

Os preprocessamentos específicos de cada abordagem serão implementados posteriormente, pois Logistic Regression e LightGBM possuem requisitos diferentes de representação das features.

### 3. Princípios de preparação

A preparação dos dados seguirá os seguintes princípios:

- **Reprodutibilidade:** as mesmas regras devem produzir os mesmos resultados quando o notebook for executado novamente.
- **Separação das amostras:** nenhuma informação de `validation` ou `test` será utilizada para ajustar transformações.
- **Rastreabilidade:** features criadas, valores especiais e decisões de tratamento serão documentados.
- **Consistência:** as mesmas transformações serão aplicadas às três amostras.
- **Interpretabilidade:** as features criadas deverão possuir significado compreensível no contexto de crédito.
- **Preservação da informação:** missing, categorias especiais e valores extremos não serão descartados sem avaliação.
- **Compatibilidade:** a base preparada deverá permitir a construção tanto do modelo tradicional quanto do modelo challenger.

## Configuração do Ambiente

Esta seção prepara o Google Colab para execução do notebook, instala as dependências do projeto, importa as bibliotecas e configura os caminhos utilizados para leitura e persistência dos dados.

O código do projeto será carregado do GitHub, enquanto os dados e artefatos intermediários poderão utilizar o Google Drive como cache.

In [1]:
# Inicialização
import os
import sys
import subprocess
from pathlib import Path
import importlib


IN_COLAB = "google.colab" in sys.modules

REPOSITORY_URL = (
    "https://github.com/angeruzzi/home-credit-default-risk.git"
)

if IN_COLAB:
    PROJECT_ROOT = Path(
        "/content/home-credit-default-risk"
    )

    if not (PROJECT_ROOT / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                REPOSITORY_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    else:
        subprocess.run(
            [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "pull",
                "--ff-only",
                "origin",
                "main",
            ],
            check=True,
        )

else:
    current_directory = Path.cwd().resolve()

    PROJECT_ROOT = (
        current_directory.parent
        if current_directory.name == "notebooks"
        else current_directory
    )

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print(f"Raiz do projeto: {PROJECT_ROOT}")

# Instalação das dependências
REQUIREMENTS_PATH = (
    PROJECT_ROOT / "requirements.txt"
)

if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {REQUIREMENTS_PATH}"
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "-r",
        str(REQUIREMENTS_PATH),
    ],
    check=True,
)

print("Dependências instaladas com sucesso.")


import utils
importlib.reload(utils)

from utils import setup_project

paths = setup_project(
    use_google_drive_cache=True,
)

Raiz do projeto: /content/home-credit-default-risk
Dependências instaladas com sucesso.
Mounted at /content/drive
Ambiente Colab: True
Modo de armazenamento: Google Drive
Raiz do projeto: /content/home-credit-default-risk
Dados: /content/drive/MyDrive/home-credit-default-risk/data
Artefatos: /content/drive/MyDrive/home-credit-default-risk/artifacts
Relatórios: /content/home-credit-default-risk/reports


In [2]:
# Imports
import json
import platform

import numpy as np
import pandas as pd
import pyarrow
import sklearn

from IPython.display import display
from sklearn.model_selection import train_test_split

## Carga e Validação dos Dados

A tabela `application_train.csv` será utilizada como fonte da base supervisionada.

Antes da preparação, serão verificadas as condições mínimas necessárias para a continuidade do pipeline:

- existência do arquivo;
- presença do identificador `SK_ID_CURR`;
- presença do `TARGET`;
- ausência de valores ausentes no `TARGET`;
- domínio binário do `TARGET`;
- unicidade do identificador;
- ausência de registros completamente duplicados.

Essas verificações funcionam como contratos de entrada e interrompem a execução caso a base não apresente a estrutura esperada.

In [3]:
# Leitura da tabela principal
APPLICATION_TRAIN_PATH = (
    paths.data_raw
    / "application_train.csv"
)

if not APPLICATION_TRAIN_PATH.exists():
    raise FileNotFoundError(
        "Arquivo application_train.csv "
        "não encontrado em: "
        f"{APPLICATION_TRAIN_PATH}"
    )

application_train = pd.read_csv(
    APPLICATION_TRAIN_PATH
)

print(
    "application_train carregado com sucesso."
)

print(
    "Dimensões:",
    application_train.shape,
)

print(
    "Origem:",
    APPLICATION_TRAIN_PATH,
)

application_train carregado com sucesso.
Dimensões: (307511, 122)
Origem: /content/drive/MyDrive/home-credit-default-risk/data/raw/application_train.csv


In [4]:
# Validação estrutural
ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"

REQUIRED_COLUMNS = {
    ID_COLUMN,
    TARGET_COLUMN,
}

missing_required_columns = (
    REQUIRED_COLUMNS
    - set(application_train.columns)
)

if missing_required_columns:
    raise ValueError(
        "Colunas obrigatórias ausentes: "
        f"{sorted(missing_required_columns)}"
    )

print(
    "Colunas obrigatórias encontradas:",
    sorted(REQUIRED_COLUMNS),
)

Colunas obrigatórias encontradas: ['SK_ID_CURR', 'TARGET']


In [5]:
# Validação do identificador
n_records = len(application_train)

n_missing_ids = (
    application_train[ID_COLUMN]
    .isna()
    .sum()
)

n_unique_ids = (
    application_train[ID_COLUMN]
    .nunique(dropna=True)
)

n_duplicated_ids = (
    application_train[ID_COLUMN]
    .duplicated(keep=False)
    .sum()
)

if n_missing_ids > 0:
    raise ValueError(
        f"{ID_COLUMN} possui "
        f"{n_missing_ids} valores ausentes."
    )

if n_unique_ids != n_records:
    raise ValueError(
        f"{ID_COLUMN} não é único. "
        f"Registros: {n_records}; "
        f"identificadores únicos: {n_unique_ids}; "
        f"registros envolvidos em duplicidade: "
        f"{n_duplicated_ids}."
    )

print(
    f"{ID_COLUMN} validado: "
    f"{n_unique_ids:,} identificadores únicos."
)

SK_ID_CURR validado: 307,511 identificadores únicos.


In [6]:
# Validação do TARGET
n_missing_target = (
    application_train[TARGET_COLUMN]
    .isna()
    .sum()
)

target_values = set(
    application_train[TARGET_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

EXPECTED_TARGET_VALUES = {0, 1}

if n_missing_target > 0:
    raise ValueError(
        f"{TARGET_COLUMN} possui "
        f"{n_missing_target} valores ausentes."
    )

if target_values != EXPECTED_TARGET_VALUES:
    raise ValueError(
        f"Domínio inesperado para {TARGET_COLUMN}: "
        f"{sorted(target_values)}. "
        "Valores esperados: [0, 1]."
    )

print(
    f"{TARGET_COLUMN} validado: "
    f"{sorted(target_values)}."
)

TARGET validado: [0, 1].


In [7]:
# Verificação de registros duplicados
n_fully_duplicated_records = (
    application_train
    .duplicated()
    .sum()
)

if n_fully_duplicated_records > 0:
    raise ValueError(
        "Foram encontrados "
        f"{n_fully_duplicated_records} "
        "registros completamente duplicados."
    )

print(
    "Registros completamente duplicados:",
    n_fully_duplicated_records,
)

Registros completamente duplicados: 0


In [8]:
# Resumo da validação
target_distribution = (
    application_train[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="n_records")
)

target_distribution["proportion"] = (
    target_distribution["n_records"]
    / len(application_train)
)

data_validation_summary = pd.DataFrame(
    [
        {
            "validation": "Número de registros",
            "value": len(application_train),
            "status": "OK",
        },
        {
            "validation": "Número de colunas",
            "value": application_train.shape[1],
            "status": "OK",
        },
        {
            "validation": "IDs ausentes",
            "value": n_missing_ids,
            "status": "OK",
        },
        {
            "validation": "IDs duplicados",
            "value": n_duplicated_ids,
            "status": "OK",
        },
        {
            "validation": "TARGET ausente",
            "value": n_missing_target,
            "status": "OK",
        },
        {
            "validation": (
                "Registros completamente duplicados"
            ),
            "value": n_fully_duplicated_records,
            "status": "OK",
        },
    ]
)

display(data_validation_summary)

display(
    target_distribution.style.format(
        {
            "n_records": "{:,.0f}",
            "proportion": "{:.2%}",
        }
    )
)

,validation,value,status
0,Número de registros,307511,OK
1,Número de colunas,122,OK
2,IDs ausentes,0,OK
3,IDs duplicados,0,OK
4,TARGET ausente,0,OK
5,Registros completamente duplicados,0,OK


,target,n_records,proportion
0,0,"282,686",91.93%
1,1,"24,825",8.07%


In [9]:
# Persistência
data_validation_output = (
    paths.metadata
    / "application_train_input_validation.csv"
)

target_distribution_output = (
    paths.metadata
    / "application_train_target_distribution.csv"
)

data_validation_summary.to_csv(
    data_validation_output,
    index=False,
)

target_distribution.to_csv(
    target_distribution_output,
    index=False,
)

print(
    "Validação salva em:",
    data_validation_output,
)

print(
    "Distribuição do TARGET salva em:",
    target_distribution_output,
)

Validação salva em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_train_input_validation.csv
Distribuição do TARGET salva em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_train_target_distribution.csv


## Divisão das Amostras

A tabela `application_train` será dividida em três amostras mutuamente exclusivas:

| Amostra | Proporção | Finalidade |
|---|---:|---|
| `train` | 70% | Ajuste dos preprocessadores e treinamento dos modelos |
| `validation` | 15% | Comparação de abordagens, seleção de features e ajuste de hiperparâmetros |
| `test` | 15% | Avaliação final do modelo selecionado |

A divisão será estratificada pelo `TARGET`, preservando aproximadamente a mesma proporção de eventos nas três amostras.

A amostra de teste deverá permanecer isolada durante o desenvolvimento. Ela será utilizada somente após a definição das transformações, features, modelo e hiperparâmetros.

Essa divisão é aleatória e estratificada, não temporal. Portanto, não representa uma validação `Out-of-Time`.

In [10]:
# Parâmetros do Split
RANDOM_STATE = 42

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

if not np.isclose(
    TRAIN_SIZE
    + VALIDATION_SIZE
    + TEST_SIZE,
    1.0,
):
    raise ValueError(
        "As proporções das amostras "
        "devem somar 1."
    )

In [11]:
# Primeira divisão: train e amostra temporária
train_data, temporary_data = (
    train_test_split(
        application_train,
        test_size=(
            VALIDATION_SIZE
            + TEST_SIZE
        ),
        random_state=RANDOM_STATE,
        stratify=application_train[
            TARGET_COLUMN
        ],
    )
)

print(
    "Train:",
    train_data.shape,
)

print(
    "Amostra temporária:",
    temporary_data.shape,
)

Train: (215257, 122)
Amostra temporária: (92254, 122)


In [12]:
# Segunda divisão: validation e test
relative_test_size = (
    TEST_SIZE
    / (
        VALIDATION_SIZE
        + TEST_SIZE
    )
)

validation_data, test_data = (
    train_test_split(
        temporary_data,
        test_size=relative_test_size,
        random_state=RANDOM_STATE,
        stratify=temporary_data[
            TARGET_COLUMN
        ],
    )
)

del temporary_data

print(
    "Train:",
    train_data.shape,
)

print(
    "Validation:",
    validation_data.shape,
)

print(
    "Test:",
    test_data.shape,
)

Train: (215257, 122)
Validation: (46127, 122)
Test: (46127, 122)


In [13]:
# Padronização dos Índices
train_data = (
    train_data
    .reset_index(drop=True)
)

validation_data = (
    validation_data
    .reset_index(drop=True)
)

test_data = (
    test_data
    .reset_index(drop=True)
)

In [14]:
# Validação das proporções e do event rate
def build_sample_summary(
    sample_name,
    dataframe,
):
    return {
        "sample": sample_name,
        "n_records": len(dataframe),
        "population_share": (
            len(dataframe)
            / len(application_train)
        ),
        "n_events": (
            dataframe[TARGET_COLUMN]
            .sum()
        ),
        "event_rate": (
            dataframe[TARGET_COLUMN]
            .mean()
        ),
    }


sample_summary = pd.DataFrame(
    [
        build_sample_summary(
            "train",
            train_data,
        ),
        build_sample_summary(
            "validation",
            validation_data,
        ),
        build_sample_summary(
            "test",
            test_data,
        ),
    ]
)

display(
    sample_summary.style.format(
        {
            "n_records": "{:,.0f}",
            "population_share": "{:.2%}",
            "n_events": "{:,.0f}",
            "event_rate": "{:.2%}",
        }
    )
)

,sample,n_records,population_share,n_events,event_rate
0,train,"215,257",70.00%,"17,377",8.07%
1,validation,"46,127",15.00%,"3,724",8.07%
2,test,"46,127",15.00%,"3,724",8.07%


In [15]:
# Diferença do event rate em relação à base completa
global_event_rate = (
    application_train[TARGET_COLUMN]
    .mean()
)

sample_summary[
    "event_rate_difference"
] = (
    sample_summary["event_rate"]
    - global_event_rate
)

display(
    sample_summary.style.format(
        {
            "n_records": "{:,.0f}",
            "population_share": "{:.2%}",
            "n_events": "{:,.0f}",
            "event_rate": "{:.2%}",
            "event_rate_difference": "{:+.3%}",
        }
    )
)

,sample,n_records,population_share,n_events,event_rate,event_rate_difference
0,train,"215,257",70.00%,"17,377",8.07%,-0.000%
1,validation,"46,127",15.00%,"3,724",8.07%,+0.000%
2,test,"46,127",15.00%,"3,724",8.07%,+0.000%


In [16]:
# Verificação de independência das amostras
train_ids = set(
    train_data[ID_COLUMN]
)

validation_ids = set(
    validation_data[ID_COLUMN]
)

test_ids = set(
    test_data[ID_COLUMN]
)

train_validation_overlap = (
    train_ids & validation_ids
)

train_test_overlap = (
    train_ids & test_ids
)

validation_test_overlap = (
    validation_ids & test_ids
)

if train_validation_overlap:
    raise ValueError(
        "Existem IDs simultaneamente em "
        "train e validation."
    )

if train_test_overlap:
    raise ValueError(
        "Existem IDs simultaneamente em "
        "train e test."
    )

if validation_test_overlap:
    raise ValueError(
        "Existem IDs simultaneamente em "
        "validation e test."
    )

all_split_ids = (
    train_ids
    | validation_ids
    | test_ids
)

original_ids = set(
    application_train[ID_COLUMN]
)

if all_split_ids != original_ids:
    raise ValueError(
        "Os IDs das três amostras não "
        "correspondem à base original."
    )

print(
    "Amostras validadas:"
    "\n- nenhum ID compartilhado;"
    "\n- todos os IDs originais preservados."
)

Amostras validadas:
- nenhum ID compartilhado;
- todos os IDs originais preservados.


In [17]:
# Persistência da atribuição das amostras
split_assignment = pd.concat(
    [
        train_data[
            [ID_COLUMN, TARGET_COLUMN]
        ].assign(sample="train"),

        validation_data[
            [ID_COLUMN, TARGET_COLUMN]
        ].assign(sample="validation"),

        test_data[
            [ID_COLUMN, TARGET_COLUMN]
        ].assign(sample="test"),
    ],
    ignore_index=True,
)

split_assignment = (
    split_assignment
    .sort_values(ID_COLUMN)
    .reset_index(drop=True)
)

display(split_assignment.head())

,SK_ID_CURR,TARGET,sample
0,100002,1,train
1,100003,0,train
2,100004,0,train
3,100006,0,validation
4,100007,0,train


In [18]:
split_assignment_output = (
    paths.data_interim
    / "application_split_assignment.parquet"
)

split_assignment.to_parquet(
    split_assignment_output,
    index=False,
)

print(
    "Atribuição das amostras salva em:",
    split_assignment_output,
)

Atribuição das amostras salva em: /content/drive/MyDrive/home-credit-default-risk/data/interim/application_split_assignment.parquet


In [19]:
split_metadata = {
    "source": "application_train.csv",
    "identifier": ID_COLUMN,
    "target": TARGET_COLUMN,
    "random_state": RANDOM_STATE,
    "stratified": True,
    "temporal_split": False,
    "proportions": {
        "train": TRAIN_SIZE,
        "validation": VALIDATION_SIZE,
        "test": TEST_SIZE,
    },
    "samples": (
        sample_summary.to_dict(
            orient="records"
        )
    ),
}

split_metadata_output = (
    paths.metadata
    / "application_split_metadata.json"
)

with open(
    split_metadata_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Metadados do split salvos em:",
    split_metadata_output,
)

Metadados do split salvos em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_split_metadata.json


## Definição do Schema de Features

As colunas da base serão separadas em três papéis:

- **identificador:** utilizado para rastrear cada aplicação, mas não como entrada dos modelos;
- **target:** variável resposta supervisionada;
- **features:** informações candidatas utilizadas para estimar o risco.

As features também serão classificadas conforme seu tipo:

- categóricas;
- numéricas binárias;
- numéricas de baixa cardinalidade;
- numéricas gerais.

A cardinalidade será calculada exclusivamente na amostra de treino. Essa classificação apoiará a definição dos preprocessamentos específicos de cada modelo.

In [20]:
# Separação dos Papéis
identifier_columns = [
    ID_COLUMN
]

target_columns = [
    TARGET_COLUMN
]

feature_columns = [
    column
    for column in train_data.columns
    if column not in (
        identifier_columns
        + target_columns
    )
]

print(
    "Identificadores:",
    identifier_columns,
)

print(
    "Target:",
    target_columns,
)

print(
    "Quantidade de features:",
    len(feature_columns),
)

Identificadores: ['SK_ID_CURR']
Target: ['TARGET']
Quantidade de features: 120


In [21]:
# Consistência das colunas entre as amostras
reference_columns = list(
    train_data.columns
)

for sample_name, dataframe in {
    "validation": validation_data,
    "test": test_data,
}.items():
    if list(dataframe.columns) != reference_columns:
        missing_columns = (
            set(reference_columns)
            - set(dataframe.columns)
        )

        extra_columns = (
            set(dataframe.columns)
            - set(reference_columns)
        )

        raise ValueError(
            f"Schema inconsistente em {sample_name}. "
            f"Ausentes: {sorted(missing_columns)}; "
            f"extras: {sorted(extra_columns)}."
        )

print(
    "As três amostras possuem "
    "as mesmas colunas e a mesma ordem."
)

As três amostras possuem as mesmas colunas e a mesma ordem.


In [22]:
# Validação dos Tipos
dtype_validation = []

for column in reference_columns:
    train_dtype = str(
        train_data[column].dtype
    )

    validation_dtype = str(
        validation_data[column].dtype
    )

    test_dtype = str(
        test_data[column].dtype
    )

    dtype_validation.append(
        {
            "column": column,
            "train_dtype": train_dtype,
            "validation_dtype": (
                validation_dtype
            ),
            "test_dtype": test_dtype,
            "consistent": (
                train_dtype
                == validation_dtype
                == test_dtype
            ),
        }
    )

dtype_validation = pd.DataFrame(
    dtype_validation
)

if not dtype_validation[
    "consistent"
].all():
    display(
        dtype_validation.query(
            "not consistent"
        )
    )

    raise ValueError(
        "Foram encontrados tipos diferentes "
        "entre as amostras."
    )

print(
    "Os tipos das colunas são consistentes."
)

Os tipos das colunas são consistentes.


In [23]:
# Classificação por Tipo
categorical_feature_columns = (
    train_data[feature_columns]
    .select_dtypes(
        include=[
            "object",
            "category",
            "string",
        ]
    )
    .columns
    .tolist()
)

numeric_feature_columns = (
    train_data[feature_columns]
    .select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)

boolean_feature_columns = (
    train_data[feature_columns]
    .select_dtypes(
        include=["bool"]
    )
    .columns
    .tolist()
)

classified_feature_columns = set(
    categorical_feature_columns
    + numeric_feature_columns
    + boolean_feature_columns
)

unclassified_feature_columns = (
    set(feature_columns)
    - classified_feature_columns
)

if unclassified_feature_columns:
    raise ValueError(
        "Features com tipos não classificados: "
        f"{sorted(unclassified_feature_columns)}"
    )

print(
    "Categóricas:",
    len(categorical_feature_columns),
)

print(
    "Numéricas:",
    len(numeric_feature_columns),
)

print(
    "Booleanas:",
    len(boolean_feature_columns),
)

Categóricas: 16
Numéricas: 104
Booleanas: 0


In [24]:
# Cardinalidade calculada em train
LOW_CARDINALITY_THRESHOLD = 20

train_feature_cardinality = (
    train_data[feature_columns]
    .nunique(dropna=True)
)

binary_numeric_feature_columns = (
    train_feature_cardinality[
        train_feature_cardinality.index.isin(
            numeric_feature_columns
        )
        & train_feature_cardinality.le(2)
    ]
    .index
    .tolist()
)

low_cardinality_numeric_feature_columns = (
    train_feature_cardinality[
        train_feature_cardinality.index.isin(
            numeric_feature_columns
        )
        & train_feature_cardinality.gt(2)
        & train_feature_cardinality.le(
            LOW_CARDINALITY_THRESHOLD
        )
    ]
    .index
    .tolist()
)

general_numeric_feature_columns = (
    train_feature_cardinality[
        train_feature_cardinality.index.isin(
            numeric_feature_columns
        )
        & train_feature_cardinality.gt(
            LOW_CARDINALITY_THRESHOLD
        )
    ]
    .index
    .tolist()
)

In [25]:
# Variáveis constantes ou totalmente ausentes
all_missing_feature_columns = (
    train_feature_cardinality[
        train_feature_cardinality.eq(0)
    ]
    .index
    .tolist()
)

constant_feature_columns = (
    train_feature_cardinality[
        train_feature_cardinality.eq(1)
    ]
    .index
    .tolist()
)

print(
    "Features totalmente ausentes em train:",
    len(all_missing_feature_columns),
)

print(
    "Features constantes em train:",
    len(constant_feature_columns),
)

if all_missing_feature_columns:
    print(
        "\nTotalmente ausentes:",
        all_missing_feature_columns,
    )

if constant_feature_columns:
    print(
        "\nConstantes:",
        constant_feature_columns,
    )

Features totalmente ausentes em train: 0
Features constantes em train: 0


In [26]:
# Construção do Schema
def classify_feature(column):
    if column in categorical_feature_columns:
        return "categorical"

    if column in boolean_feature_columns:
        return "boolean"

    if column in binary_numeric_feature_columns:
        return "binary_numeric"

    if (
        column
        in low_cardinality_numeric_feature_columns
    ):
        return "low_cardinality_numeric"

    if column in general_numeric_feature_columns:
        return "general_numeric"

    return "unclassified"


feature_schema = pd.DataFrame(
    {
        "feature": feature_columns,
    }
)

feature_schema["dtype"] = (
    feature_schema["feature"]
    .map(
        lambda column: str(
            train_data[column].dtype
        )
    )
)

feature_schema["feature_type"] = (
    feature_schema["feature"]
    .map(classify_feature)
)

feature_schema["n_unique_train"] = (
    feature_schema["feature"]
    .map(train_feature_cardinality)
)

feature_schema["missing_rate_train"] = (
    feature_schema["feature"]
    .map(
        train_data[feature_columns]
        .isna()
        .mean()
    )
)

feature_schema["all_missing_train"] = (
    feature_schema["feature"]
    .isin(all_missing_feature_columns)
)

feature_schema["constant_train"] = (
    feature_schema["feature"]
    .isin(constant_feature_columns)
)

feature_schema = (
    feature_schema
    .sort_values(
        [
            "feature_type",
            "feature",
        ]
    )
    .reset_index(drop=True)
)

display(
    feature_schema.style.format(
        {
            "n_unique_train": "{:,.0f}",
            "missing_rate_train": "{:.2%}",
        }
    )
)

,feature,dtype,feature_type,n_unique_train,missing_rate_train,all_missing_train,constant_train
0,FLAG_CONT_MOBILE,int64,binary_numeric,2,0.00%,False,False
1,FLAG_DOCUMENT_10,int64,binary_numeric,2,0.00%,False,False
2,FLAG_DOCUMENT_11,int64,binary_numeric,2,0.00%,False,False
3,FLAG_DOCUMENT_12,int64,binary_numeric,2,0.00%,False,False
4,FLAG_DOCUMENT_13,int64,binary_numeric,2,0.00%,False,False
5,FLAG_DOCUMENT_14,int64,binary_numeric,2,0.00%,False,False
6,FLAG_DOCUMENT_15,int64,binary_numeric,2,0.00%,False,False
7,FLAG_DOCUMENT_16,int64,binary_numeric,2,0.00%,False,False
8,FLAG_DOCUMENT_17,int64,binary_numeric,2,0.00%,False,False
9,FLAG_DOCUMENT_18,int64,binary_numeric,2,0.00%,False,False


In [27]:
# Validação da Classificação
if (
    feature_schema["feature_type"]
    == "unclassified"
).any():
    raise ValueError(
        "Existem features sem classificação."
    )

if feature_schema["feature"].duplicated().any():
    raise ValueError(
        "Existem features duplicadas no schema."
    )

if set(feature_schema["feature"]) != set(
    feature_columns
):
    raise ValueError(
        "O schema não contempla todas as features."
    )

print(
    "Schema validado:",
    len(feature_schema),
    "features classificadas.",
)

Schema validado: 120 features classificadas.


In [28]:
# Persistência do Schema
feature_schema_output = (
    paths.metadata
    / "application_feature_schema.csv"
)

feature_schema.to_csv(
    feature_schema_output,
    index=False,
)

print(
    "Schema de features salvo em:",
    feature_schema_output,
)

Schema de features salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_feature_schema.csv


In [29]:
feature_schema_metadata = {
    "identifier_columns": identifier_columns,
    "target_columns": target_columns,
    "feature_columns": feature_columns,
    "categorical_features": (
        categorical_feature_columns
    ),
    "boolean_features": (
        boolean_feature_columns
    ),
    "binary_numeric_features": (
        binary_numeric_feature_columns
    ),
    "low_cardinality_numeric_features": (
        low_cardinality_numeric_feature_columns
    ),
    "general_numeric_features": (
        general_numeric_feature_columns
    ),
    "all_missing_train": (
        all_missing_feature_columns
    ),
    "constant_train": (
        constant_feature_columns
    ),
    "low_cardinality_threshold": (
        LOW_CARDINALITY_THRESHOLD
    ),
}

feature_schema_metadata_output = (
    paths.metadata
    / "application_feature_schema.json"
)

with open(
    feature_schema_metadata_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_schema_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Metadados do schema salvos em:",
    feature_schema_metadata_output,
)

Metadados do schema salvos em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_feature_schema.json


## Feature Engineering Determinístico

Nesta etapa serão implementadas transformações definidas por regras fixas de negócio e pelas evidências da EDA.

As features criadas serão agrupadas em:

- tratamento de códigos especiais;
- variáveis temporais em unidades interpretáveis;
- razões financeiras;
- características da composição familiar;
- agregações dos scores externos.

Essas transformações não utilizam o `TARGET`, médias, medianas, quantis ou qualquer parâmetro aprendido da população. Portanto, podem ser aplicadas de forma idêntica a `train`, `validation` e `test`.

Imputação, agrupamento de categorias, `binning`, encoding e scaling não serão realizados nesta etapa.

In [30]:
# Divisão Segura
def safe_divide(
    numerator,
    denominator,
):
    """
    Divide duas Series somente quando o
    denominador é finito e maior que zero.
    """

    numerator = pd.to_numeric(
        numerator,
        errors="coerce",
    )

    denominator = pd.to_numeric(
        denominator,
        errors="coerce",
    )

    valid_denominator = (
        denominator.notna()
        & np.isfinite(denominator)
        & denominator.gt(0)
    )

    result = pd.Series(
        np.nan,
        index=numerator.index,
        dtype="float64",
    )

    result.loc[valid_denominator] = (
        numerator.loc[valid_denominator]
        / denominator.loc[valid_denominator]
    )

    return result

In [31]:
# Função de criação das features
SPECIAL_DAYS_EMPLOYED_VALUE = 365243
DAYS_PER_YEAR = 365.25


def create_application_features(
    dataframe,
):
    """
    Cria features determinísticas para a
    tabela de aplicações.

    A função não utiliza o TARGET e não
    aprende parâmetros a partir dos dados.
    """

    required_columns = {
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
        "DAYS_LAST_PHONE_CHANGE",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_GOODS_PRICE",
        "CNT_CHILDREN",
        "CNT_FAM_MEMBERS",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            "Colunas necessárias para "
            "feature engineering ausentes: "
            f"{sorted(missing_columns)}"
        )

    result = dataframe.copy()

    # Código especial de DAYS_EMPLOYED
    result[
        "DAYS_EMPLOYED_SPECIAL"
    ] = (
        result["DAYS_EMPLOYED"]
        .eq(SPECIAL_DAYS_EMPLOYED_VALUE)
        .astype("int8")
    )

    result.loc[
        result["DAYS_EMPLOYED"].eq(
            SPECIAL_DAYS_EMPLOYED_VALUE
        ),
        "DAYS_EMPLOYED",
    ] = np.nan

    # Variáveis temporais
    result["AGE_YEARS"] = (
        -result["DAYS_BIRTH"]
        / DAYS_PER_YEAR
    )

    result["EMPLOYMENT_YEARS"] = (
        -result["DAYS_EMPLOYED"]
        / DAYS_PER_YEAR
    )

    result["REGISTRATION_YEARS"] = (
        -result["DAYS_REGISTRATION"]
        / DAYS_PER_YEAR
    )

    result["ID_PUBLISH_YEARS"] = (
        -result["DAYS_ID_PUBLISH"]
        / DAYS_PER_YEAR
    )

    result["LAST_PHONE_CHANGE_YEARS"] = (
        -result["DAYS_LAST_PHONE_CHANGE"]
        / DAYS_PER_YEAR
    )

    result[
        "EMPLOYMENT_AGE_RATIO"
    ] = safe_divide(
        result["EMPLOYMENT_YEARS"],
        result["AGE_YEARS"],
    )

    # Razões financeiras
    result[
        "CREDIT_INCOME_RATIO"
    ] = safe_divide(
        result["AMT_CREDIT"],
        result["AMT_INCOME_TOTAL"],
    )

    result[
        "ANNUITY_INCOME_RATIO"
    ] = safe_divide(
        result["AMT_ANNUITY"],
        result["AMT_INCOME_TOTAL"],
    )

    result[
        "CREDIT_ANNUITY_RATIO"
    ] = safe_divide(
        result["AMT_CREDIT"],
        result["AMT_ANNUITY"],
    )

    result[
        "CREDIT_GOODS_RATIO"
    ] = safe_divide(
        result["AMT_CREDIT"],
        result["AMT_GOODS_PRICE"],
    )

    result[
        "INCOME_PER_PERSON"
    ] = safe_divide(
        result["AMT_INCOME_TOTAL"],
        result["CNT_FAM_MEMBERS"],
    )

    # Composição familiar
    result[
        "CHILDREN_FAMILY_RATIO"
    ] = safe_divide(
        result["CNT_CHILDREN"],
        result["CNT_FAM_MEMBERS"],
    )

    result[
        "ADULT_FAMILY_MEMBERS"
    ] = (
        result["CNT_FAM_MEMBERS"]
        - result["CNT_CHILDREN"]
    )

    # Agregação dos scores externos
    external_source_columns = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    ]

    result[
        "EXT_SOURCE_MEAN"
    ] = (
        result[external_source_columns]
        .mean(axis=1)
    )

    result[
        "EXT_SOURCE_MIN"
    ] = (
        result[external_source_columns]
        .min(axis=1)
    )

    result[
        "EXT_SOURCE_MAX"
    ] = (
        result[external_source_columns]
        .max(axis=1)
    )

    result[
        "EXT_SOURCE_STD"
    ] = (
        result[external_source_columns]
        .std(
            axis=1,
            ddof=0,
        )
    )

    result[
        "EXT_SOURCE_MISSING_COUNT"
    ] = (
        result[external_source_columns]
        .isna()
        .sum(axis=1)
        .astype("int8")
    )

    return result

In [32]:
# Aplicação nas três amostras
train_prepared = (
    create_application_features(
        train_data
    )
)

validation_prepared = (
    create_application_features(
        validation_data
    )
)

test_prepared = (
    create_application_features(
        test_data
    )
)

print(
    "Train:",
    train_prepared.shape,
)

print(
    "Validation:",
    validation_prepared.shape,
)

print(
    "Test:",
    test_prepared.shape,
)

Train: (215257, 141)
Validation: (46127, 141)
Test: (46127, 141)


In [33]:
# Identificação das novas features
engineered_feature_columns = [
    column
    for column in train_prepared.columns
    if column not in train_data.columns
]

print(
    "Quantidade de features criadas:",
    len(engineered_feature_columns),
)

for column in engineered_feature_columns:
    print(f"- {column}")

Quantidade de features criadas: 19
- DAYS_EMPLOYED_SPECIAL
- AGE_YEARS
- EMPLOYMENT_YEARS
- REGISTRATION_YEARS
- ID_PUBLISH_YEARS
- LAST_PHONE_CHANGE_YEARS
- EMPLOYMENT_AGE_RATIO
- CREDIT_INCOME_RATIO
- ANNUITY_INCOME_RATIO
- CREDIT_ANNUITY_RATIO
- CREDIT_GOODS_RATIO
- INCOME_PER_PERSON
- CHILDREN_FAMILY_RATIO
- ADULT_FAMILY_MEMBERS
- EXT_SOURCE_MEAN
- EXT_SOURCE_MIN
- EXT_SOURCE_MAX
- EXT_SOURCE_STD
- EXT_SOURCE_MISSING_COUNT


In [34]:
# Validação das transformações
prepared_samples = {
    "train": train_prepared,
    "validation": validation_prepared,
    "test": test_prepared,
}

reference_prepared_columns = list(
    train_prepared.columns
)

feature_engineering_validation = []

for sample_name, dataframe in (
    prepared_samples.items()
):
    same_columns = (
        list(dataframe.columns)
        == reference_prepared_columns
    )

    n_infinite_values = (
        dataframe[
            engineered_feature_columns
        ]
        .select_dtypes(
            include=[np.number]
        )
        .isin([np.inf, -np.inf])
        .sum()
        .sum()
    )

    unchanged_ids = (
        dataframe[ID_COLUMN]
        .equals(
            {
                "train": train_data,
                "validation": validation_data,
                "test": test_data,
            }[sample_name][ID_COLUMN]
        )
    )

    unchanged_target = (
        dataframe[TARGET_COLUMN]
        .equals(
            {
                "train": train_data,
                "validation": validation_data,
                "test": test_data,
            }[sample_name][TARGET_COLUMN]
        )
    )

    feature_engineering_validation.append(
        {
            "sample": sample_name,
            "n_records": len(dataframe),
            "n_columns": dataframe.shape[1],
            "same_columns": same_columns,
            "n_infinite_values": (
                n_infinite_values
            ),
            "unchanged_ids": unchanged_ids,
            "unchanged_target": (
                unchanged_target
            ),
        }
    )

feature_engineering_validation = (
    pd.DataFrame(
        feature_engineering_validation
    )
)

display(feature_engineering_validation)

,sample,n_records,n_columns,same_columns,n_infinite_values,unchanged_ids,unchanged_target
0,train,215257,141,True,0,True,True
1,validation,46127,141,True,0,True,True
2,test,46127,141,True,0,True,True


In [35]:
if not feature_engineering_validation[
    "same_columns"
].all():
    raise ValueError(
        "As amostras preparadas possuem "
        "colunas diferentes."
    )

if (
    feature_engineering_validation[
        "n_infinite_values"
    ]
    .gt(0)
    .any()
):
    raise ValueError(
        "Foram produzidos valores infinitos."
    )

if not feature_engineering_validation[
    "unchanged_ids"
].all():
    raise ValueError(
        "Os identificadores foram alterados."
    )

if not feature_engineering_validation[
    "unchanged_target"
].all():
    raise ValueError(
        "O TARGET foi alterado."
    )

print(
    "Feature engineering validado "
    "nas três amostras."
)

Feature engineering validado nas três amostras.


In [36]:
# Verificação do código especial
days_employed_check = pd.DataFrame(
    [
        {
            "sample": sample_name,
            "n_special_flag": (
                dataframe[
                    "DAYS_EMPLOYED_SPECIAL"
                ].sum()
            ),
            "n_original_special_remaining": (
                dataframe["DAYS_EMPLOYED"]
                .eq(
                    SPECIAL_DAYS_EMPLOYED_VALUE
                )
                .sum()
            ),
            "n_days_employed_missing": (
                dataframe["DAYS_EMPLOYED"]
                .isna()
                .sum()
            ),
        }
        for sample_name, dataframe
        in prepared_samples.items()
    ]
)

display(days_employed_check)

,sample,n_special_flag,n_original_special_remaining,n_days_employed_missing
0,train,38756,0,38756
1,validation,8277,0,8277
2,test,8341,0,8341


In [37]:
# Registro das features criadas
engineered_feature_metadata = pd.DataFrame(
    [
        {
            "feature": (
                "DAYS_EMPLOYED_SPECIAL"
            ),
            "group": "special_value",
            "description": (
                "Flag para o código especial "
                "365243 em DAYS_EMPLOYED."
            ),
        },
        {
            "feature": "AGE_YEARS",
            "group": "temporal",
            "description": (
                "Idade aproximada em anos."
            ),
        },
        {
            "feature": "EMPLOYMENT_YEARS",
            "group": "temporal",
            "description": (
                "Tempo aproximado de emprego "
                "em anos."
            ),
        },
        {
            "feature": "REGISTRATION_YEARS",
            "group": "temporal",
            "description": (
                "Tempo aproximado desde a "
                "alteração do registro."
            ),
        },
        {
            "feature": "ID_PUBLISH_YEARS",
            "group": "temporal",
            "description": (
                "Tempo aproximado desde a "
                "emissão do documento."
            ),
        },
        {
            "feature": (
                "LAST_PHONE_CHANGE_YEARS"
            ),
            "group": "temporal",
            "description": (
                "Tempo aproximado desde a "
                "última alteração do telefone."
            ),
        },
        {
            "feature": (
                "EMPLOYMENT_AGE_RATIO"
            ),
            "group": "temporal",
            "description": (
                "Tempo de emprego em relação "
                "à idade."
            ),
        },
        {
            "feature": (
                "CREDIT_INCOME_RATIO"
            ),
            "group": "financial",
            "description": (
                "Crédito em relação à renda."
            ),
        },
        {
            "feature": (
                "ANNUITY_INCOME_RATIO"
            ),
            "group": "financial",
            "description": (
                "Anuidade em relação à renda."
            ),
        },
        {
            "feature": (
                "CREDIT_ANNUITY_RATIO"
            ),
            "group": "financial",
            "description": (
                "Crédito em relação à anuidade."
            ),
        },
        {
            "feature": (
                "CREDIT_GOODS_RATIO"
            ),
            "group": "financial",
            "description": (
                "Crédito em relação ao valor "
                "do bem."
            ),
        },
        {
            "feature": "INCOME_PER_PERSON",
            "group": "financial",
            "description": (
                "Renda aproximada por "
                "integrante da família."
            ),
        },
        {
            "feature": (
                "CHILDREN_FAMILY_RATIO"
            ),
            "group": "family",
            "description": (
                "Proporção de filhos no "
                "grupo familiar."
            ),
        },
        {
            "feature": (
                "ADULT_FAMILY_MEMBERS"
            ),
            "group": "family",
            "description": (
                "Quantidade aproximada de "
                "adultos na família."
            ),
        },
        {
            "feature": "EXT_SOURCE_MEAN",
            "group": "external_score",
            "description": (
                "Média dos scores externos "
                "disponíveis."
            ),
        },
        {
            "feature": "EXT_SOURCE_MIN",
            "group": "external_score",
            "description": (
                "Menor score externo disponível."
            ),
        },
        {
            "feature": "EXT_SOURCE_MAX",
            "group": "external_score",
            "description": (
                "Maior score externo disponível."
            ),
        },
        {
            "feature": "EXT_SOURCE_STD",
            "group": "external_score",
            "description": (
                "Dispersão dos scores externos."
            ),
        },
        {
            "feature": (
                "EXT_SOURCE_MISSING_COUNT"
            ),
            "group": "external_score",
            "description": (
                "Quantidade de scores externos "
                "ausentes."
            ),
        },
    ]
)

display(engineered_feature_metadata)

,feature,group,description
0,DAYS_EMPLOYED_SPECIAL,special_value,Flag para o código especial 365243 em DAYS_EMP...
1,AGE_YEARS,temporal,Idade aproximada em anos.
2,EMPLOYMENT_YEARS,temporal,Tempo aproximado de emprego em anos.
3,REGISTRATION_YEARS,temporal,Tempo aproximado desde a alteração do registro.
4,ID_PUBLISH_YEARS,temporal,Tempo aproximado desde a emissão do documento.
5,LAST_PHONE_CHANGE_YEARS,temporal,Tempo aproximado desde a última alteração do t...
6,EMPLOYMENT_AGE_RATIO,temporal,Tempo de emprego em relação à idade.
7,CREDIT_INCOME_RATIO,financial,Crédito em relação à renda.
8,ANNUITY_INCOME_RATIO,financial,Anuidade em relação à renda.
9,CREDIT_ANNUITY_RATIO,financial,Crédito em relação à anuidade.


In [38]:
# Persitência
engineered_feature_metadata_output = (
    paths.metadata
    / "application_engineered_features.csv"
)

engineered_feature_metadata.to_csv(
    engineered_feature_metadata_output,
    index=False,
)

print(
    "Metadados das features salvos em:",
    engineered_feature_metadata_output,
)

Metadados das features salvos em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_engineered_features.csv


## Validação das Features Criadas

A criação de uma feature pode produzir valores matematicamente válidos, mas incompatíveis com seu significado de negócio.

Serão verificadas condições como:

- idade não positiva ou excessivamente elevada;
- tempo de emprego negativo ou superior à idade;
- intervalos temporais negativos após a conversão;
- quantidade negativa de adultos na família;
- proporções familiares fora do intervalo esperado;
- scores externos fora do intervalo entre `0` e `1`;
- razões financeiras negativas;
- presença de valores infinitos.

Valores ausentes não serão tratados como inconsistências, pois podem decorrer legitimamente da indisponibilidade das informações de origem.

In [39]:
consistency_rules = {
    "AGE_NOT_POSITIVE": (
        lambda dataframe:
        dataframe["AGE_YEARS"].le(0)
    ),
    "AGE_ABOVE_100": (
        lambda dataframe:
        dataframe["AGE_YEARS"].gt(100)
    ),
    "EMPLOYMENT_NEGATIVE": (
        lambda dataframe:
        dataframe["EMPLOYMENT_YEARS"].lt(0)
    ),
    "EMPLOYMENT_ABOVE_AGE": (
        lambda dataframe:
        dataframe["EMPLOYMENT_YEARS"].gt(
            dataframe["AGE_YEARS"]
        )
    ),
    "REGISTRATION_NEGATIVE": (
        lambda dataframe:
        dataframe["REGISTRATION_YEARS"].lt(0)
    ),
    "ID_PUBLISH_NEGATIVE": (
        lambda dataframe:
        dataframe["ID_PUBLISH_YEARS"].lt(0)
    ),
    "PHONE_CHANGE_NEGATIVE": (
        lambda dataframe:
        dataframe[
            "LAST_PHONE_CHANGE_YEARS"
        ].lt(0)
    ),
    "ADULT_FAMILY_MEMBERS_NEGATIVE": (
        lambda dataframe:
        dataframe[
            "ADULT_FAMILY_MEMBERS"
        ].lt(0)
    ),
    "CHILDREN_FAMILY_RATIO_BELOW_ZERO": (
        lambda dataframe:
        dataframe[
            "CHILDREN_FAMILY_RATIO"
        ].lt(0)
    ),
    "CHILDREN_FAMILY_RATIO_ABOVE_ONE": (
        lambda dataframe:
        dataframe[
            "CHILDREN_FAMILY_RATIO"
        ].gt(1)
    ),
    "EMPLOYMENT_AGE_RATIO_BELOW_ZERO": (
        lambda dataframe:
        dataframe[
            "EMPLOYMENT_AGE_RATIO"
        ].lt(0)
    ),
    "EMPLOYMENT_AGE_RATIO_ABOVE_ONE": (
        lambda dataframe:
        dataframe[
            "EMPLOYMENT_AGE_RATIO"
        ].gt(1)
    ),
    "FINANCIAL_RATIO_NEGATIVE": (
        lambda dataframe:
        dataframe[
            [
                "CREDIT_INCOME_RATIO",
                "ANNUITY_INCOME_RATIO",
                "CREDIT_ANNUITY_RATIO",
                "CREDIT_GOODS_RATIO",
                "INCOME_PER_PERSON",
            ]
        ]
        .lt(0)
        .any(axis=1)
    ),
    "EXT_SOURCE_OUTSIDE_ZERO_ONE": (
        lambda dataframe:
        (
            dataframe[
                [
                    "EXT_SOURCE_1",
                    "EXT_SOURCE_2",
                    "EXT_SOURCE_3",
                ]
            ]
            .lt(0)
            .any(axis=1)
            |
            dataframe[
                [
                    "EXT_SOURCE_1",
                    "EXT_SOURCE_2",
                    "EXT_SOURCE_3",
                ]
            ]
            .gt(1)
            .any(axis=1)
        )
    ),
}

In [40]:
# Aplicação das regras
consistency_results = []
consistency_masks = {}

for sample_name, dataframe in (
    prepared_samples.items()
):
    consistency_masks[sample_name] = {}

    for rule_name, rule_function in (
        consistency_rules.items()
    ):
        rule_mask = (
            rule_function(dataframe)
            .fillna(False)
            .astype(bool)
        )

        consistency_masks[
            sample_name
        ][rule_name] = rule_mask

        n_violations = int(
            rule_mask.sum()
        )

        consistency_results.append(
            {
                "sample": sample_name,
                "rule": rule_name,
                "n_violations": n_violations,
                "violation_rate": (
                    n_violations
                    / len(dataframe)
                ),
            }
        )

consistency_results = pd.DataFrame(
    consistency_results
)

display(
    consistency_results.style.format(
        {
            "n_violations": "{:,.0f}",
            "violation_rate": "{:.4%}",
        }
    )
)

,sample,rule,n_violations,violation_rate
0,train,AGE_NOT_POSITIVE,0,0.0000%
1,train,AGE_ABOVE_100,0,0.0000%
2,train,EMPLOYMENT_NEGATIVE,0,0.0000%
3,train,EMPLOYMENT_ABOVE_AGE,0,0.0000%
4,train,REGISTRATION_NEGATIVE,0,0.0000%
5,train,ID_PUBLISH_NEGATIVE,0,0.0000%
6,train,PHONE_CHANGE_NEGATIVE,0,0.0000%
7,train,ADULT_FAMILY_MEMBERS_NEGATIVE,0,0.0000%
8,train,CHILDREN_FAMILY_RATIO_BELOW_ZERO,0,0.0000%
9,train,CHILDREN_FAMILY_RATIO_ABOVE_ONE,0,0.0000%


In [41]:
# Resumo por Amostra
consistency_sample_summary = (
    consistency_results
    .groupby("sample")
    .agg(
        n_rules=("rule", "nunique"),
        n_rules_with_violations=(
            "n_violations",
            lambda values: values.gt(0).sum(),
        ),
        total_rule_violations=(
            "n_violations",
            "sum",
        ),
    )
    .reset_index()
)

display(consistency_sample_summary)

,sample,n_rules,n_rules_with_violations,total_rule_violations
0,test,14,0,0
1,train,14,0,0
2,validation,14,0,0


In [42]:
# Regras Com violação
violated_rules = (
    consistency_results
    .query("n_violations > 0")
    .sort_values(
        [
            "n_violations",
            "sample",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

if violated_rules.empty:
    print(
        "Nenhuma inconsistência foi "
        "identificada pelas regras definidas."
    )
else:
    display(
        violated_rules.style.format(
            {
                "n_violations": "{:,.0f}",
                "violation_rate": "{:.4%}",
            }
        )
    )

Nenhuma inconsistência foi identificada pelas regras definidas.


In [43]:
# Inspeção dos registros inconsistentes
consistency_inspection_columns = [
    ID_COLUMN,
    TARGET_COLUMN,
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "EMPLOYMENT_AGE_RATIO",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "ADULT_FAMILY_MEMBERS",
    "CHILDREN_FAMILY_RATIO",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

train_inconsistency_mask = pd.Series(
    False,
    index=train_prepared.index,
)

for rule_mask in consistency_masks[
    "train"
].values():
    train_inconsistency_mask = (
        train_inconsistency_mask
        | rule_mask
    )

train_inconsistent_records = (
    train_prepared.loc[
        train_inconsistency_mask,
        consistency_inspection_columns,
    ]
)

print(
    "Clientes de train com pelo menos "
    "uma inconsistência:",
    len(train_inconsistent_records),
)

display(
    train_inconsistent_records.head(20)
)

Clientes de train com pelo menos uma inconsistência: 0


,SK_ID_CURR,TARGET,DAYS_BIRTH,DAYS_EMPLOYED,AGE_YEARS,EMPLOYMENT_YEARS,EMPLOYMENT_AGE_RATIO,CNT_CHILDREN,CNT_FAM_MEMBERS,ADULT_FAMILY_MEMBERS,CHILDREN_FAMILY_RATIO,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3


In [44]:
# Perfil das Features Criadas
engineered_feature_profile = []

for sample_name, dataframe in (
    prepared_samples.items()
):
    for column in engineered_feature_columns:
        series = dataframe[column]

        engineered_feature_profile.append(
            {
                "sample": sample_name,
                "feature": column,
                "dtype": str(series.dtype),
                "n_non_missing": (
                    series.notna().sum()
                ),
                "n_missing": (
                    series.isna().sum()
                ),
                "missing_rate": (
                    series.isna().mean()
                ),
                "n_unique": (
                    series.nunique(
                        dropna=True
                    )
                ),
                "min": (
                    series.min()
                    if pd.api.types
                    .is_numeric_dtype(series)
                    else np.nan
                ),
                "median": (
                    series.median()
                    if pd.api.types
                    .is_numeric_dtype(series)
                    else np.nan
                ),
                "p99": (
                    series.quantile(0.99)
                    if pd.api.types
                    .is_numeric_dtype(series)
                    else np.nan
                ),
                "max": (
                    series.max()
                    if pd.api.types
                    .is_numeric_dtype(series)
                    else np.nan
                ),
            }
        )

engineered_feature_profile = pd.DataFrame(
    engineered_feature_profile
)

display(
    engineered_feature_profile.style.format(
        {
            "n_non_missing": "{:,.0f}",
            "n_missing": "{:,.0f}",
            "missing_rate": "{:.2%}",
            "n_unique": "{:,.0f}",
            "min": "{:,.4f}",
            "median": "{:,.4f}",
            "p99": "{:,.4f}",
            "max": "{:,.4f}",
        }
    )
)

,sample,feature,dtype,n_non_missing,n_missing,missing_rate,n_unique,min,median,p99,max
0,train,DAYS_EMPLOYED_SPECIAL,int8,"215,257",0,0.00%,2,0.0000,0.0000,1.0000,1.0000
1,train,AGE_YEARS,float64,"215,257",0,0.00%,"17,377",21.0075,43.1075,66.8172,68.9938
2,train,EMPLOYMENT_YEARS,float64,"176,501","38,756",18.00%,"11,769",-0.0000,4.5147,31.0308,48.1396
3,train,REGISTRATION_YEARS,float64,"215,257",0,0.00%,"15,249",-0.0000,12.3532,38.0765,67.5483
4,train,ID_PUBLISH_YEARS,float64,"215,257",0,0.00%,"6,122",0.0000,8.9254,14.9103,19.7043
5,train,LAST_PHONE_CHANGE_YEARS,float64,"215,256",1,0.00%,"3,720",-0.0000,2.0780,8.6379,11.7509
6,train,EMPLOYMENT_AGE_RATIO,float64,"176,501","38,756",18.00%,"175,499",-0.0000,0.1187,0.5909,0.7186
7,train,CREDIT_INCOME_RATIO,float64,"215,257",0,0.00%,"39,372",0.0048,3.2651,13.0435,49.2272
8,train,ANNUITY_INCOME_RATIO,float64,"215,249",8,0.00%,"71,916",0.0002,0.1626,0.4849,1.4516
9,train,CREDIT_ANNUITY_RATIO,float64,"215,249",8,0.00%,"33,434",8.0367,20.0000,37.7942,45.3051


### Interpretação da validação

As regras de consistência representam critérios de plausibilidade, e não evidência automática de erro.

Caso sejam identificadas violações, os registros deverão ser analisados antes da definição do tratamento. As alternativas incluem:

- preservar o valor quando houver justificativa;
- converter o valor inconsistente em missing;
- criar uma flag indicadora;
- aplicar limites definidos por regras de negócio;
- excluir a feature, caso sua qualidade seja insuficiente.

Nenhum cliente será removido automaticamente por uma violação isolada. Em modelos de crédito, a exclusão de registros altera a população de desenvolvimento e deve possuir uma justificativa clara e documentada.

In [45]:
# Persistência
consistency_results_output = (
    paths.metadata
    / "application_feature_consistency.csv"
)

engineered_feature_profile_output = (
    paths.metadata
    / "application_engineered_feature_profile.csv"
)

consistency_results.to_csv(
    consistency_results_output,
    index=False,
)

engineered_feature_profile.to_csv(
    engineered_feature_profile_output,
    index=False,
)

print(
    "Validação de consistência salva em:",
    consistency_results_output,
)

print(
    "Perfil das features salvo em:",
    engineered_feature_profile_output,
)

Validação de consistência salva em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_feature_consistency.csv
Perfil das features salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_engineered_feature_profile.csv


## Definição da Base Analítica

A base analítica comum deverá preservar o máximo possível da informação disponível, mas algumas colunas não serão utilizadas como features:

- identificadores;
- colunas totalmente ausentes na amostra de treino;
- colunas constantes na amostra de treino;
- variáveis temporais originais que foram substituídas por versões equivalentes em anos.

A remoção das variáveis temporais originais não implica perda de informação, pois as novas versões são transformações lineares das mesmas medidas. Mantê-las simultaneamente produziria redundância perfeita.

Nenhuma feature será removida nesta etapa somente por apresentar:

- alta proporção de missing;
- correlação baixa com o `TARGET`;
- correlação elevada com outra feature;
- PSI elevado;
- valores extremos.

Esses critérios serão avaliados posteriormente dentro dos pipelines de modelagem e validação.

In [52]:
# Reavaliação de constantes após o feature engineering
prepared_feature_columns = [
    column
    for column in train_prepared.columns
    if column not in [
        ID_COLUMN,
        TARGET_COLUMN,
    ]
]

prepared_train_cardinality = (
    train_prepared[
        prepared_feature_columns
    ]
    .nunique(dropna=True)
)

all_missing_prepared_columns = (
    prepared_train_cardinality[
        prepared_train_cardinality.eq(0)
    ]
    .index
    .tolist()
)

constant_prepared_columns = (
    prepared_train_cardinality[
        prepared_train_cardinality.eq(1)
    ]
    .index
    .tolist()
)

print(
    "Totalmente ausentes em train:",
    len(all_missing_prepared_columns),
)

print(
    "Constantes em train:",
    len(constant_prepared_columns),
)

if all_missing_prepared_columns:
    print(
        "\nColunas totalmente ausentes:",
        all_missing_prepared_columns,
    )

if constant_prepared_columns:
    print(
        "\nColunas constantes:",
        constant_prepared_columns,
    )

Totalmente ausentes em train: 0
Constantes em train: 0


In [53]:
# Variáveis temporais redundantes
original_temporal_columns = [
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE",
]

temporal_replacement_mapping = {
    "DAYS_BIRTH": "AGE_YEARS",
    "DAYS_EMPLOYED": "EMPLOYMENT_YEARS",
    "DAYS_REGISTRATION": (
        "REGISTRATION_YEARS"
    ),
    "DAYS_ID_PUBLISH": (
        "ID_PUBLISH_YEARS"
    ),
    "DAYS_LAST_PHONE_CHANGE": (
        "LAST_PHONE_CHANGE_YEARS"
    ),
}

temporal_replacement_table = (
    pd.DataFrame(
        [
            {
                "original_feature": original,
                "replacement_feature": replacement,
                "reason": (
                    "Transformação para anos"
                ),
            }
            for original, replacement
            in temporal_replacement_mapping.items()
        ]
    )
)

display(temporal_replacement_table)

,original_feature,replacement_feature,reason
0,DAYS_BIRTH,AGE_YEARS,Transformação para anos
1,DAYS_EMPLOYED,EMPLOYMENT_YEARS,Transformação para anos
2,DAYS_REGISTRATION,REGISTRATION_YEARS,Transformação para anos
3,DAYS_ID_PUBLISH,ID_PUBLISH_YEARS,Transformação para anos
4,DAYS_LAST_PHONE_CHANGE,LAST_PHONE_CHANGE_YEARS,Transformação para anos


In [54]:
# Validação se todas as substituições existem:
missing_original_temporal = [
    column
    for column in original_temporal_columns
    if column not in train_prepared.columns
]

missing_replacement_temporal = [
    column
    for column in (
        temporal_replacement_mapping.values()
    )
    if column not in train_prepared.columns
]

if missing_original_temporal:
    raise ValueError(
        "Variáveis temporais originais ausentes: "
        f"{missing_original_temporal}"
    )

if missing_replacement_temporal:
    raise ValueError(
        "Variáveis temporais substitutas ausentes: "
        f"{missing_replacement_temporal}"
    )

In [55]:
# Registro das exclusões
feature_exclusions = []

for column in all_missing_prepared_columns:
    feature_exclusions.append(
        {
            "feature": column,
            "reason": (
                "Totally missing in train"
            ),
            "reason_pt": (
                "Totalmente ausente em train"
            ),
        }
    )

for column in constant_prepared_columns:
    feature_exclusions.append(
        {
            "feature": column,
            "reason": (
                "Constant in train"
            ),
            "reason_pt": (
                "Constante em train"
            ),
        }
    )

for column in original_temporal_columns:
    feature_exclusions.append(
        {
            "feature": column,
            "reason": (
                "Replaced by equivalent "
                "feature in years"
            ),
            "reason_pt": (
                "Substituída por feature "
                "equivalente em anos"
            ),
        }
    )

feature_exclusions = (
    pd.DataFrame(feature_exclusions)
    .drop_duplicates(
        subset=["feature"],
        keep="first",
    )
    .sort_values("feature")
    .reset_index(drop=True)
)

display(feature_exclusions)

,feature,reason,reason_pt
0,DAYS_BIRTH,Replaced by equivalent feature in years,Substituída por feature equivalente em anos
1,DAYS_EMPLOYED,Replaced by equivalent feature in years,Substituída por feature equivalente em anos
2,DAYS_ID_PUBLISH,Replaced by equivalent feature in years,Substituída por feature equivalente em anos
3,DAYS_LAST_PHONE_CHANGE,Replaced by equivalent feature in years,Substituída por feature equivalente em anos
4,DAYS_REGISTRATION,Replaced by equivalent feature in years,Substituída por feature equivalente em anos


In [56]:
# Lista final de features candidatas
excluded_feature_columns = (
    feature_exclusions["feature"]
    .tolist()
)

model_feature_columns = [
    column
    for column in prepared_feature_columns
    if column not in excluded_feature_columns
]

print(
    "Features após feature engineering:",
    len(prepared_feature_columns),
)

print(
    "Features excluídas:",
    len(excluded_feature_columns),
)

print(
    "Features candidatas finais:",
    len(model_feature_columns),
)

Features após feature engineering: 139
Features excluídas: 5
Features candidatas finais: 134


In [57]:
# Construção das bases analíticas
def build_modeling_base(
    dataframe,
    feature_columns,
):
    selected_columns = (
        [ID_COLUMN, TARGET_COLUMN]
        + feature_columns
    )

    missing_columns = (
        set(selected_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            "Colunas ausentes na construção "
            "da base analítica: "
            f"{sorted(missing_columns)}"
        )

    return (
        dataframe[selected_columns]
        .copy()
    )

In [58]:
train_modeling = build_modeling_base(
    train_prepared,
    model_feature_columns,
)

validation_modeling = build_modeling_base(
    validation_prepared,
    model_feature_columns,
)

test_modeling = build_modeling_base(
    test_prepared,
    model_feature_columns,
)

modeling_samples = {
    "train": train_modeling,
    "validation": validation_modeling,
    "test": test_modeling,
}

for sample_name, dataframe in (
    modeling_samples.items()
):
    print(
        sample_name,
        dataframe.shape,
    )

train (215257, 136)
validation (46127, 136)
test (46127, 136)


## Separação entre Identificador, Features e Target

O identificador será preservado para rastreabilidade, mas não será fornecido aos modelos.

Para cada amostra serão mantidos:

- `ids`: identificadores das aplicações;
- `X`: matriz de features;
- `y`: variável resposta.

Essa separação reduz o risco de utilizar acidentalmente `SK_ID_CURR` ou `TARGET` como feature.

In [59]:
train_ids = train_modeling[
    ID_COLUMN
].copy()

validation_ids = validation_modeling[
    ID_COLUMN
].copy()

test_ids = test_modeling[
    ID_COLUMN
].copy()

X_train = train_modeling[
    model_feature_columns
].copy()

X_validation = validation_modeling[
    model_feature_columns
].copy()

X_test = test_modeling[
    model_feature_columns
].copy()

y_train = train_modeling[
    TARGET_COLUMN
].copy()

y_validation = validation_modeling[
    TARGET_COLUMN
].copy()

y_test = test_modeling[
    TARGET_COLUMN
].copy()

In [60]:
# Validação da separação
if ID_COLUMN in X_train.columns:
    raise ValueError(
        f"{ID_COLUMN} foi incluído em X_train."
    )

if TARGET_COLUMN in X_train.columns:
    raise ValueError(
        f"{TARGET_COLUMN} foi incluído em X_train."
    )

if list(X_train.columns) != list(
    X_validation.columns
):
    raise ValueError(
        "X_train e X_validation possuem "
        "features diferentes."
    )

if list(X_train.columns) != list(
    X_test.columns
):
    raise ValueError(
        "X_train e X_test possuem "
        "features diferentes."
    )

if not (
    len(X_train)
    == len(y_train)
    == len(train_ids)
):
    raise ValueError(
        "Dimensões inconsistentes em train."
    )

if not (
    len(X_validation)
    == len(y_validation)
    == len(validation_ids)
):
    raise ValueError(
        "Dimensões inconsistentes "
        "em validation."
    )

if not (
    len(X_test)
    == len(y_test)
    == len(test_ids)
):
    raise ValueError(
        "Dimensões inconsistentes em test."
    )

print(
    "Separação entre ID, X e y validada."
)

Separação entre ID, X e y validada.


In [61]:
# Resumo das bases
modeling_base_summary = pd.DataFrame(
    [
        {
            "sample": sample_name,
            "n_records": len(dataframe),
            "n_features": (
                dataframe.shape[1] - 2
            ),
            "event_rate": (
                dataframe[TARGET_COLUMN]
                .mean()
            ),
            "memory_mb": (
                dataframe
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        }
        for sample_name, dataframe
        in modeling_samples.items()
    ]
)

display(
    modeling_base_summary.style.format(
        {
            "n_records": "{:,.0f}",
            "n_features": "{:,.0f}",
            "event_rate": "{:.2%}",
            "memory_mb": "{:,.2f}",
        }
    )
)

,sample,n_records,n_features,event_rate,memory_mb
0,train,"215,257",134,8.07%,373.62
1,validation,"46,127",134,8.07%,80.06
2,test,"46,127",134,8.07%,80.05


## Persistência das Bases Processadas

As bases serão armazenadas em formato Parquet, que preserva os tipos das colunas, ocupa menos espaço e geralmente apresenta leitura mais eficiente que CSV.

Os arquivos permanecerão no Google Drive utilizado como cache e não serão versionados no GitHub.

In [92]:
# Persistência das bases processadas
processed_output_paths = {
    "train": (
        paths.data_processed
        / "application_train_prepared.parquet"
    ),
    "validation": (
        paths.data_processed
        / "application_validation_prepared.parquet"
    ),
    "test": (
        paths.data_processed
        / "application_holdout_test_prepared.parquet"
    ),
}

for sample_name, dataframe in (
    modeling_samples.items()
):
    output_path = (
        processed_output_paths[
            sample_name
        ]
    )

    dataframe.to_parquet(
        output_path,
        index=False,
    )

    print(
        f"{sample_name} salvo em: "
        f"{output_path}"
    )

train salvo em: /content/drive/MyDrive/home-credit-default-risk/data/processed/application_train_prepared.parquet
validation salvo em: /content/drive/MyDrive/home-credit-default-risk/data/processed/application_validation_prepared.parquet
test salvo em: /content/drive/MyDrive/home-credit-default-risk/data/processed/application_holdout_test_prepared.parquet


In [93]:
# Validação dos arquivos persistidos
persisted_files_validation = []

for sample_name, expected_dataframe in (
    modeling_samples.items()
):
    output_path = (
        processed_output_paths[
            sample_name
        ]
    )

    persisted_dataframe = pd.read_parquet(
        output_path
    )

    persisted_files_validation.append(
        {
            "sample": sample_name,
            "file_exists": (
                output_path.exists()
            ),
            "expected_rows": (
                len(expected_dataframe)
            ),
            "persisted_rows": (
                len(persisted_dataframe)
            ),
            "expected_columns": (
                expected_dataframe.shape[1]
            ),
            "persisted_columns": (
                persisted_dataframe.shape[1]
            ),
            "same_columns": (
                list(
                    expected_dataframe.columns
                )
                == list(
                    persisted_dataframe.columns
                )
            ),
            "file_size_mb": (
                output_path.stat().st_size
                / 1024**2
            ),
        }
    )

persisted_files_validation = pd.DataFrame(
    persisted_files_validation
)

display(
    persisted_files_validation.style.format(
        {
            "expected_rows": "{:,.0f}",
            "persisted_rows": "{:,.0f}",
            "expected_columns": "{:,.0f}",
            "persisted_columns": "{:,.0f}",
            "file_size_mb": "{:,.2f}",
        }
    )
)

,sample,file_exists,expected_rows,persisted_rows,expected_columns,persisted_columns,same_columns,file_size_mb
0,train,True,"215,257","215,257",136,136,True,26.27
1,validation,True,"46,127","46,127",136,136,True,6.51
2,test,True,"46,127","46,127",136,136,True,6.52


In [94]:
# Interrupção se houver divergências
if not (
    persisted_files_validation[
        "file_exists"
    ].all()
    and (
        persisted_files_validation[
            "expected_rows"
        ]
        == persisted_files_validation[
            "persisted_rows"
        ]
    ).all()
    and (
        persisted_files_validation[
            "expected_columns"
        ]
        == persisted_files_validation[
            "persisted_columns"
        ]
    ).all()
    and persisted_files_validation[
        "same_columns"
    ].all()
):
    raise ValueError(
        "Os arquivos persistidos não "
        "correspondem às bases em memória."
    )

print(
    "Arquivos processados validados."
)

Arquivos processados validados.


In [95]:
# Persistência das exclusões
feature_exclusions_output = (
    paths.metadata
    / "application_feature_exclusions.csv"
)

feature_exclusions.to_csv(
    feature_exclusions_output,
    index=False,
)

model_features_output = (
    paths.metadata
    / "application_model_features.json"
)

with open(
    model_features_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "n_features": len(
                model_feature_columns
            ),
            "features": (
                model_feature_columns
            ),
            "excluded_features": (
                feature_exclusions
                .to_dict(
                    orient="records"
                )
            ),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Exclusões registradas em:",
    feature_exclusions_output,
)

print(
    "Lista de features salva em:",
    model_features_output,
)

Exclusões registradas em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_feature_exclusions.csv
Lista de features salva em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_model_features.json


## Compatibilidade das Variáveis Categóricas

Uma categoria presente em `validation` ou `test`, mas ausente em `train`, é denominada categoria desconhecida ou `unknown`.

Como os preprocessadores serão ajustados exclusivamente em `train`, eles precisam possuir uma regra explícita para esses casos. Caso contrário, o encoding poderá falhar durante a validação ou em uma aplicação futura.

Também serão identificadas categorias raras na amostra de treino. Categorias com poucas observações podem gerar:

- estimativas instáveis de `event rate`;
- valores extremos de WoE;
- coeficientes instáveis na Logistic Regression;
- maior risco de overfitting.

Nesta etapa, nenhuma categoria será agrupada. Serão apenas registrados os grupos que os pipelines precisarão tratar.

In [96]:
# Identificação das features categóricas
categorical_model_features = (
    X_train
    .select_dtypes(
        include=[
            "object",
            "category",
            "string",
        ]
    )
    .columns
    .tolist()
)

print(
    "Quantidade de features categóricas:",
    len(categorical_model_features),
)

for feature in categorical_model_features:
    print(f"- {feature}")

Quantidade de features categóricas: 16
- NAME_CONTRACT_TYPE
- CODE_GENDER
- FLAG_OWN_CAR
- FLAG_OWN_REALTY
- NAME_TYPE_SUITE
- NAME_INCOME_TYPE
- NAME_EDUCATION_TYPE
- NAME_FAMILY_STATUS
- NAME_HOUSING_TYPE
- OCCUPATION_TYPE
- WEEKDAY_APPR_PROCESS_START
- ORGANIZATION_TYPE
- FONDKAPREMONT_MODE
- HOUSETYPE_MODE
- WALLSMATERIAL_MODE
- EMERGENCYSTATE_MODE


In [97]:
# Parâmetros para categorias raras
RARE_CATEGORY_MIN_SHARE = 0.005
RARE_CATEGORY_MIN_COUNT = 100

print(
    "Participação mínima:",
    f"{RARE_CATEGORY_MIN_SHARE:.2%}",
)

print(
    "Quantidade mínima:",
    RARE_CATEGORY_MIN_COUNT,
)

Participação mínima: 0.50%
Quantidade mínima: 100


In [98]:
# Função de padronização para análise
MISSING_CATEGORY_LABEL = "<MISSING>"


def normalize_category_for_analysis(
    series,
):
    """
    Converte categorias para string e representa
    valores ausentes de forma explícita.
    """

    return (
        series
        .astype("string")
        .fillna(MISSING_CATEGORY_LABEL)
    )

In [99]:
# Construção do perfil detalhado
categorical_compatibility_details = []

for feature in categorical_model_features:
    train_categories = (
        normalize_category_for_analysis(
            X_train[feature]
        )
    )

    validation_categories = (
        normalize_category_for_analysis(
            X_validation[feature]
        )
    )

    test_categories = (
        normalize_category_for_analysis(
            X_test[feature]
        )
    )

    train_counts = (
        train_categories
        .value_counts(dropna=False)
    )

    validation_counts = (
        validation_categories
        .value_counts(dropna=False)
    )

    test_counts = (
        test_categories
        .value_counts(dropna=False)
    )

    all_categories = (
        train_counts.index
        .union(validation_counts.index)
        .union(test_counts.index)
    )

    for category in all_categories:
        train_count = int(
            train_counts.get(
                category,
                0,
            )
        )

        validation_count = int(
            validation_counts.get(
                category,
                0,
            )
        )

        test_count = int(
            test_counts.get(
                category,
                0,
            )
        )

        train_share = (
            train_count / len(X_train)
        )

        categorical_compatibility_details.append(
            {
                "feature": feature,
                "category": str(category),
                "train_count": train_count,
                "train_share": train_share,
                "validation_count": (
                    validation_count
                ),
                "validation_share": (
                    validation_count
                    / len(X_validation)
                ),
                "test_count": test_count,
                "test_share": (
                    test_count
                    / len(X_test)
                ),
                "absent_from_train": (
                    train_count == 0
                ),
                "rare_in_train": (
                    train_count > 0
                    and (
                        train_share
                        < RARE_CATEGORY_MIN_SHARE
                        or train_count
                        < RARE_CATEGORY_MIN_COUNT
                    )
                ),
            }
        )

categorical_compatibility_details = (
    pd.DataFrame(
        categorical_compatibility_details
    )
)

In [100]:
# Resumo da compatibilidade por feature
categorical_compatibility_summary = []

for feature in categorical_model_features:
    feature_details = (
        categorical_compatibility_details
        .query("feature == @feature")
    )

    validation_unknown_mask = (
        normalize_category_for_analysis(
            X_validation[feature]
        )
        .isin(
            feature_details.loc[
                feature_details[
                    "absent_from_train"
                ],
                "category",
            ]
        )
    )

    test_unknown_mask = (
        normalize_category_for_analysis(
            X_test[feature]
        )
        .isin(
            feature_details.loc[
                feature_details[
                    "absent_from_train"
                ],
                "category",
            ]
        )
    )

    categorical_compatibility_summary.append(
        {
            "feature": feature,
            "n_categories_train": (
                feature_details[
                    "train_count"
                ]
                .gt(0)
                .sum()
            ),
            "n_rare_categories_train": (
                feature_details[
                    "rare_in_train"
                ]
                .sum()
            ),
            "n_unknown_categories_validation": (
                feature_details.loc[
                    feature_details[
                        "validation_count"
                    ].gt(0),
                    "absent_from_train",
                ]
                .sum()
            ),
            "unknown_records_validation": (
                validation_unknown_mask.sum()
            ),
            "unknown_rate_validation": (
                validation_unknown_mask.mean()
            ),
            "n_unknown_categories_test": (
                feature_details.loc[
                    feature_details[
                        "test_count"
                    ].gt(0),
                    "absent_from_train",
                ]
                .sum()
            ),
            "unknown_records_test": (
                test_unknown_mask.sum()
            ),
            "unknown_rate_test": (
                test_unknown_mask.mean()
            ),
        }
    )

categorical_compatibility_summary = (
    pd.DataFrame(
        categorical_compatibility_summary
    )
    .sort_values(
        [
            "unknown_rate_validation",
            "unknown_rate_test",
            "n_rare_categories_train",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    categorical_compatibility_summary.style.format(
        {
            "n_categories_train": "{:,.0f}",
            "n_rare_categories_train": "{:,.0f}",
            "n_unknown_categories_validation": "{:,.0f}",
            "unknown_records_validation": "{:,.0f}",
            "unknown_rate_validation": "{:.4%}",
            "n_unknown_categories_test": "{:,.0f}",
            "unknown_records_test": "{:,.0f}",
            "unknown_rate_test": "{:.4%}",
        }
    )
)

,feature,n_categories_train,n_rare_categories_train,n_unknown_categories_validation,unknown_records_validation,unknown_rate_validation,n_unknown_categories_test,unknown_records_test,unknown_rate_test
0,ORGANIZATION_TYPE,58,29,0,0,0.0000%,0,0,0.0000%
1,OCCUPATION_TYPE,19,5,0,0,0.0000%,0,0,0.0000%
2,NAME_INCOME_TYPE,8,4,0,0,0.0000%,0,0,0.0000%
3,NAME_TYPE_SUITE,8,3,0,0,0.0000%,0,0,0.0000%
4,HOUSETYPE_MODE,4,2,0,0,0.0000%,0,0,0.0000%
5,CODE_GENDER,3,1,0,0,0.0000%,0,0,0.0000%
6,NAME_EDUCATION_TYPE,5,1,0,0,0.0000%,0,0,0.0000%
7,NAME_FAMILY_STATUS,6,1,0,0,0.0000%,0,0,0.0000%
8,NAME_HOUSING_TYPE,6,1,0,0,0.0000%,0,0,0.0000%
9,NAME_CONTRACT_TYPE,2,0,0,0,0.0000%,0,0,0.0000%


In [101]:
# Identificação das categorias desconhecidas
unknown_categories = (
    categorical_compatibility_details
    .query(
        "absent_from_train "
        "and (validation_count > 0 or test_count > 0)"
    )
    .sort_values(
        [
            "feature",
            "validation_count",
            "test_count",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

if unknown_categories.empty:
    print(
        "Nenhuma categoria desconhecida foi "
        "encontrada em validation ou test."
    )
else:
    display(
        unknown_categories.style.format(
            {
                "train_count": "{:,.0f}",
                "train_share": "{:.4%}",
                "validation_count": "{:,.0f}",
                "validation_share": "{:.4%}",
                "test_count": "{:,.0f}",
                "test_share": "{:.4%}",
            }
        )
    )

Nenhuma categoria desconhecida foi encontrada em validation ou test.


In [102]:
# Identificação das categorias raras em train
rare_train_categories = (
    categorical_compatibility_details
    .query("rare_in_train")
    .sort_values(
        [
            "feature",
            "train_share",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

print(
    "Quantidade total de categorias raras:",
    len(rare_train_categories),
)

display(
    rare_train_categories.style.format(
        {
            "train_count": "{:,.0f}",
            "train_share": "{:.4%}",
            "validation_count": "{:,.0f}",
            "validation_share": "{:.4%}",
            "test_count": "{:,.0f}",
            "test_share": "{:.4%}",
        }
    )
)

Quantidade total de categorias raras: 47


,feature,category,train_count,train_share,validation_count,validation_share,test_count,test_share,absent_from_train,rare_in_train
0,CODE_GENDER,XNA,2,0.0009%,2,0.0043%,0,0.0000%,False,True
1,HOUSETYPE_MODE,terraced house,867,0.4028%,193,0.4184%,152,0.3295%,False,True
2,HOUSETYPE_MODE,specific housing,"1,041",0.4836%,238,0.5160%,220,0.4769%,False,True
3,NAME_EDUCATION_TYPE,Academic degree,104,0.0483%,33,0.0715%,27,0.0585%,False,True
4,NAME_FAMILY_STATUS,Unknown,1,0.0005%,1,0.0022%,0,0.0000%,False,True
5,NAME_HOUSING_TYPE,Co-op apartment,791,0.3675%,162,0.3512%,169,0.3664%,False,True
6,NAME_INCOME_TYPE,Maternity leave,3,0.0014%,0,0.0000%,2,0.0043%,False,True
7,NAME_INCOME_TYPE,Businessman,9,0.0042%,1,0.0022%,0,0.0000%,False,True
8,NAME_INCOME_TYPE,Student,9,0.0042%,5,0.0108%,4,0.0087%,False,True
9,NAME_INCOME_TYPE,Unemployed,16,0.0074%,4,0.0087%,2,0.0043%,False,True


### Estratégia planejada para categorias

As regras definitivas serão ajustadas exclusivamente em `train`.

Para o modelo tradicional:

- missing será tratado como grupo explícito;
- categorias raras poderão ser agrupadas;
- categorias desconhecidas serão direcionadas para um grupo específico;
- o WoE será calculado somente na amostra de treino.

Para o modelo challenger:

- as categorias serão representadas de forma compatível com o LightGBM;
- valores ausentes serão preservados ou representados explicitamente;
- categorias desconhecidas serão mapeadas para um nível reservado;
- nenhuma categoria de `validation` ou `test` será incorporada ao ajuste.

Os valores `XNA`, `XAP`, `Unknown` e semelhantes não serão automaticamente convertidos em missing. Seu tratamento dependerá do significado da variável e será documentado no respectivo pipeline.

In [103]:
# Persistência da compatibilidade categórica
categorical_compatibility_summary_output = (
    paths.metadata
    / "application_categorical_compatibility_summary.csv"
)

categorical_compatibility_details_output = (
    paths.metadata
    / "application_categorical_compatibility_details.csv"
)

rare_train_categories_output = (
    paths.metadata
    / "application_rare_train_categories.csv"
)

unknown_categories_output = (
    paths.metadata
    / "application_unknown_categories.csv"
)

categorical_compatibility_summary.to_csv(
    categorical_compatibility_summary_output,
    index=False,
)

categorical_compatibility_details.to_csv(
    categorical_compatibility_details_output,
    index=False,
)

rare_train_categories.to_csv(
    rare_train_categories_output,
    index=False,
)

unknown_categories.to_csv(
    unknown_categories_output,
    index=False,
)

print(
    "Resumo salvo em:",
    categorical_compatibility_summary_output,
)

print(
    "Detalhamento salvo em:",
    categorical_compatibility_details_output,
)

Resumo salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_categorical_compatibility_summary.csv
Detalhamento salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_categorical_compatibility_details.csv


In [104]:
# Registro dos parâmetros categóricos
categorical_handling_metadata = {
    "fit_sample": "train",
    "missing_label": (
        MISSING_CATEGORY_LABEL
    ),
    "rare_category_min_share": (
        RARE_CATEGORY_MIN_SHARE
    ),
    "rare_category_min_count": (
        RARE_CATEGORY_MIN_COUNT
    ),
    "n_categorical_features": len(
        categorical_model_features
    ),
    "categorical_features": (
        categorical_model_features
    ),
    "strategy": {
        "missing": (
            "Explicit group in model-specific pipeline"
        ),
        "rare": (
            "Group based on train frequency"
        ),
        "unknown": (
            "Reserved unknown group"
        ),
    },
}

categorical_handling_metadata_output = (
    paths.metadata
    / "application_categorical_handling.json"
)

with open(
    categorical_handling_metadata_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        categorical_handling_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Parâmetros categóricos salvos em:",
    categorical_handling_metadata_output,
)

Parâmetros categóricos salvos em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_categorical_handling.json


## Estratégia para Valores Ausentes

Os valores ausentes serão preservados na base analítica comum. A imputação antecipada poderia eliminar informação relevante e impor uma mesma decisão a modelos com características diferentes.

A estratégia dependerá da abordagem:

- no modelo tradicional, missing poderá constituir um bin próprio no processo de `binning` e WoE;
- no LightGBM, missing poderá ser tratado nativamente pelo algoritmo;
- flags adicionais serão consideradas quando a indisponibilidade da informação possuir significado próprio;
- qualquer estatística de imputação será calculada exclusivamente em `train`.

Também será verificado se alguma feature não possui missing em `train`, mas apresenta valores ausentes em `validation` ou `test`. Essa situação exige que o pipeline tenha uma regra de fallback mesmo que o missing não tenha sido observado durante o ajuste.

In [105]:
# Identificação dos tipos na base final

numeric_model_features = (
    X_train
    .select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)

categorical_model_features = (
    X_train
    .select_dtypes(
        include=[
            "object",
            "category",
            "string",
        ]
    )
    .columns
    .tolist()
)

classified_model_features = set(
    numeric_model_features
    + categorical_model_features
)

unclassified_model_features = (
    set(model_feature_columns)
    - classified_model_features
)

if unclassified_model_features:
    raise ValueError(
        "Features finais sem classificação: "
        f"{sorted(unclassified_model_features)}"
    )

print(
    "Features numéricas:",
    len(numeric_model_features),
)

print(
    "Features categóricas:",
    len(categorical_model_features),
)

Features numéricas: 118
Features categóricas: 16


In [106]:
# Construção do perfil de missing
missing_compatibility_summary = []

for feature in model_feature_columns:
    train_missing_count = (
        X_train[feature]
        .isna()
        .sum()
    )

    validation_missing_count = (
        X_validation[feature]
        .isna()
        .sum()
    )

    test_missing_count = (
        X_test[feature]
        .isna()
        .sum()
    )

    train_missing_rate = (
        train_missing_count
        / len(X_train)
    )

    validation_missing_rate = (
        validation_missing_count
        / len(X_validation)
    )

    test_missing_rate = (
        test_missing_count
        / len(X_test)
    )

    feature_type = (
        "numeric"
        if feature in numeric_model_features
        else "categorical"
    )

    missing_compatibility_summary.append(
        {
            "feature": feature,
            "feature_type": feature_type,
            "train_missing_count": (
                train_missing_count
            ),
            "train_missing_rate": (
                train_missing_rate
            ),
            "validation_missing_count": (
                validation_missing_count
            ),
            "validation_missing_rate": (
                validation_missing_rate
            ),
            "test_missing_count": (
                test_missing_count
            ),
            "test_missing_rate": (
                test_missing_rate
            ),
            "validation_train_difference": (
                validation_missing_rate
                - train_missing_rate
            ),
            "test_train_difference": (
                test_missing_rate
                - train_missing_rate
            ),
            "missing_absent_train_present_later": (
                train_missing_count == 0
                and (
                    validation_missing_count > 0
                    or test_missing_count > 0
                )
            ),
        }
    )

missing_compatibility_summary = (
    pd.DataFrame(
        missing_compatibility_summary
    )
    .sort_values(
        "train_missing_rate",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    missing_compatibility_summary.style.format(
        {
            "train_missing_count": "{:,.0f}",
            "train_missing_rate": "{:.2%}",
            "validation_missing_count": "{:,.0f}",
            "validation_missing_rate": "{:.2%}",
            "test_missing_count": "{:,.0f}",
            "test_missing_rate": "{:.2%}",
            "validation_train_difference": "{:+.2%}",
            "test_train_difference": "{:+.2%}",
        }
    )
)

,feature,feature_type,train_missing_count,train_missing_rate,validation_missing_count,validation_missing_rate,test_missing_count,test_missing_rate,validation_train_difference,test_train_difference,missing_absent_train_present_later
0,COMMONAREA_AVG,numeric,"150,300",69.82%,"32,253",69.92%,"32,312",70.05%,+0.10%,+0.23%,False
1,COMMONAREA_MODE,numeric,"150,300",69.82%,"32,253",69.92%,"32,312",70.05%,+0.10%,+0.23%,False
2,COMMONAREA_MEDI,numeric,"150,300",69.82%,"32,253",69.92%,"32,312",70.05%,+0.10%,+0.23%,False
3,NONLIVINGAPARTMENTS_AVG,numeric,"149,354",69.38%,"32,060",69.50%,"32,100",69.59%,+0.12%,+0.21%,False
4,NONLIVINGAPARTMENTS_MODE,numeric,"149,354",69.38%,"32,060",69.50%,"32,100",69.59%,+0.12%,+0.21%,False
5,NONLIVINGAPARTMENTS_MEDI,numeric,"149,354",69.38%,"32,060",69.50%,"32,100",69.59%,+0.12%,+0.21%,False
6,FONDKAPREMONT_MODE,categorical,"147,099",68.34%,"31,572",68.45%,"31,624",68.56%,+0.11%,+0.22%,False
7,LIVINGAPARTMENTS_AVG,numeric,"147,049",68.31%,"31,572",68.45%,"31,578",68.46%,+0.13%,+0.15%,False
8,LIVINGAPARTMENTS_MEDI,numeric,"147,049",68.31%,"31,572",68.45%,"31,578",68.46%,+0.13%,+0.15%,False
9,LIVINGAPARTMENTS_MODE,numeric,"147,049",68.31%,"31,572",68.45%,"31,578",68.46%,+0.13%,+0.15%,False


In [107]:
# Resumo de missing por tipo de feature
missing_by_feature_type = (
    missing_compatibility_summary
    .groupby("feature_type")
    .agg(
        n_features=(
            "feature",
            "nunique",
        ),
        n_features_with_missing_train=(
            "train_missing_count",
            lambda values: values.gt(0).sum(),
        ),
        mean_missing_rate_train=(
            "train_missing_rate",
            "mean",
        ),
        max_missing_rate_train=(
            "train_missing_rate",
            "max",
        ),
        n_features_missing_only_later=(
            "missing_absent_train_present_later",
            "sum",
        ),
    )
    .reset_index()
)

display(
    missing_by_feature_type.style.format(
        {
            "n_features": "{:,.0f}",
            "n_features_with_missing_train": "{:,.0f}",
            "mean_missing_rate_train": "{:.2%}",
            "max_missing_rate_train": "{:.2%}",
            "n_features_missing_only_later": "{:,.0f}",
        }
    )
)

,feature_type,n_features,n_features_with_missing_train,mean_missing_rate_train,max_missing_rate_train,n_features_missing_only_later
0,categorical,16,6,15.52%,68.34%,0
1,numeric,118,73,23.41%,69.82%,0


In [108]:
# Identificação de missing não observado em train
missing_only_later = (
    missing_compatibility_summary
    .query(
        "missing_absent_train_present_later"
    )
    .reset_index(drop=True)
)

if missing_only_later.empty:
    print(
        "Todas as features com missing em validation "
        "ou test também possuem missing em train."
    )
else:
    display(
        missing_only_later.style.format(
            {
                "train_missing_count": "{:,.0f}",
                "train_missing_rate": "{:.2%}",
                "validation_missing_count": "{:,.0f}",
                "validation_missing_rate": "{:.2%}",
                "test_missing_count": "{:,.0f}",
                "test_missing_rate": "{:.2%}",
            }
        )
    )

Todas as features com missing em validation ou test também possuem missing em train.


In [109]:
# Identificação das maiores diferenças de missing
missing_difference_ranking = (
    missing_compatibility_summary
    .assign(
        maximum_absolute_difference=lambda df: (
            df[
                [
                    "validation_train_difference",
                    "test_train_difference",
                ]
            ]
            .abs()
            .max(axis=1)
        )
    )
    .sort_values(
        "maximum_absolute_difference",
        ascending=False,
    )
    .head(20)
)

display(
    missing_difference_ranking.style.format(
        {
            "train_missing_rate": "{:.2%}",
            "validation_missing_rate": "{:.2%}",
            "test_missing_rate": "{:.2%}",
            "validation_train_difference": "{:+.2%}",
            "test_train_difference": "{:+.2%}",
            "maximum_absolute_difference": "{:.2%}",
        }
    )
)

,feature,feature_type,train_missing_count,train_missing_rate,validation_missing_count,validation_missing_rate,test_missing_count,test_missing_rate,validation_train_difference,test_train_difference,missing_absent_train_present_later,maximum_absolute_difference
26,NONLIVINGAREA_MODE,numeric,118577,55.09%,25491,55.26%,25614,55.53%,+0.18%,+0.44%,False,0.44%
24,NONLIVINGAREA_AVG,numeric,118577,55.09%,25491,55.26%,25614,55.53%,+0.18%,+0.44%,False,0.44%
25,NONLIVINGAREA_MEDI,numeric,118577,55.09%,25491,55.26%,25614,55.53%,+0.18%,+0.44%,False,0.44%
40,HOUSETYPE_MODE,categorical,107834,50.10%,23164,50.22%,23299,50.51%,+0.12%,+0.42%,False,0.42%
32,APARTMENTS_MEDI,numeric,109076,50.67%,23424,50.78%,23561,51.08%,+0.11%,+0.41%,False,0.41%
33,APARTMENTS_MODE,numeric,109076,50.67%,23424,50.78%,23561,51.08%,+0.11%,+0.41%,False,0.41%
31,APARTMENTS_AVG,numeric,109076,50.67%,23424,50.78%,23561,51.08%,+0.11%,+0.41%,False,0.41%
43,FLOORSMAX_MEDI,numeric,106970,49.69%,22951,49.76%,23099,50.08%,+0.06%,+0.38%,False,0.38%
42,FLOORSMAX_MODE,numeric,106970,49.69%,22951,49.76%,23099,50.08%,+0.06%,+0.38%,False,0.38%
41,FLOORSMAX_AVG,numeric,106970,49.69%,22951,49.76%,23099,50.08%,+0.06%,+0.38%,False,0.38%


### Decisões para tratamento de missing

Os valores ausentes permanecerão na base analítica comum.

#### Modelo tradicional — WoE e Logistic Regression

- missing será considerado uma categoria ou bin específico;
- o agrupamento será determinado somente com `train`;
- serão utilizados mecanismos de suavização quando um bin não possuir eventos ou não eventos;
- o mesmo mapeamento será aplicado a `validation` e `test`;
- missing não observado em `train` deverá possuir um fallback explícito.

#### Modelo challenger — LightGBM

- valores numéricos ausentes serão preservados para o tratamento nativo do algoritmo;
- features categóricas deverão possuir representação compatível com o modelo;
- categorias ausentes ou desconhecidas terão códigos reservados;
- a criação de flags adicionais será avaliada por validação.

#### Base analítica comum

Não serão aplicados nesta base:

- imputação por média ou mediana;
- `SimpleImputer`;
- WoE;
- One-Hot Encoding;
- Ordinal Encoding;
- scaling;
- normalização;
- `binning`;
- cap ou winsorization.

Essas operações deverão pertencer aos pipelines específicos, garantindo que seus parâmetros sejam ajustados exclusivamente em `train`.

In [110]:
# Registro das estratégias de preprocessing
preprocessing_strategy = pd.DataFrame(
    [
        {
            "operation": "Missing numérico",
            "common_base": "Preservar como NaN",
            "logistic_pipeline": (
                "Bin específico ou imputação "
                "ajustada em train"
            ),
            "lightgbm_pipeline": (
                "Tratamento nativo"
            ),
        },
        {
            "operation": "Missing categórico",
            "common_base": (
                "Preservar como missing"
            ),
            "logistic_pipeline": (
                "Categoria explícita"
            ),
            "lightgbm_pipeline": (
                "Categoria ou código reservado"
            ),
        },
        {
            "operation": "Categoria rara",
            "common_base": "Não agrupar",
            "logistic_pipeline": (
                "Agrupar com regra de train"
            ),
            "lightgbm_pipeline": (
                "Avaliar agrupamento por validação"
            ),
        },
        {
            "operation": (
                "Categoria desconhecida"
            ),
            "common_base": "Preservar valor",
            "logistic_pipeline": (
                "Grupo unknown"
            ),
            "lightgbm_pipeline": (
                "Código unknown"
            ),
        },
        {
            "operation": "Binning",
            "common_base": "Não aplicar",
            "logistic_pipeline": (
                "Ajustar somente em train"
            ),
            "lightgbm_pipeline": (
                "Não necessário previamente"
            ),
        },
        {
            "operation": "WoE",
            "common_base": "Não aplicar",
            "logistic_pipeline": (
                "Ajustar somente em train"
            ),
            "lightgbm_pipeline": (
                "Não aplicar"
            ),
        },
        {
            "operation": "Scaling",
            "common_base": "Não aplicar",
            "logistic_pipeline": (
                "Avaliar após WoE"
            ),
            "lightgbm_pipeline": (
                "Não necessário"
            ),
        },
        {
            "operation": (
                "Valores extremos"
            ),
            "common_base": "Preservar",
            "logistic_pipeline": (
                "Tratar por binning ou cap "
                "ajustado em train"
            ),
            "lightgbm_pipeline": (
                "Avaliar por validação"
            ),
        },
    ]
)

display(preprocessing_strategy)

,operation,common_base,logistic_pipeline,lightgbm_pipeline
0,Missing numérico,Preservar como NaN,Bin específico ou imputação ajustada em train,Tratamento nativo
1,Missing categórico,Preservar como missing,Categoria explícita,Categoria ou código reservado
2,Categoria rara,Não agrupar,Agrupar com regra de train,Avaliar agrupamento por validação
3,Categoria desconhecida,Preservar valor,Grupo unknown,Código unknown
4,Binning,Não aplicar,Ajustar somente em train,Não necessário previamente
5,WoE,Não aplicar,Ajustar somente em train,Não aplicar
6,Scaling,Não aplicar,Avaliar após WoE,Não necessário
7,Valores extremos,Preservar,Tratar por binning ou cap ajustado em train,Avaliar por validação


In [112]:
# Persistência do perfil de missing
missing_compatibility_output = (
    paths.metadata
    / "application_missing_compatibility.csv"
)

missing_compatibility_summary.to_csv(
    missing_compatibility_output,
    index=False,
)

preprocessing_strategy_output = (
    paths.metadata
    / "application_preprocessing_strategy.csv"
)

preprocessing_strategy.to_csv(
    preprocessing_strategy_output,
    index=False,
)

print(
    "Perfil de missing salvo em:",
    missing_compatibility_output,
)

print(
    "Estratégia de preprocessing salva em:",
    preprocessing_strategy_output,
)

Perfil de missing salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_missing_compatibility.csv
Estratégia de preprocessing salva em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_preprocessing_strategy.csv


In [113]:
# Registro dos metadados de missing
missing_strategy_metadata = {
    "fit_sample": "train",
    "common_base": {
        "preserve_numeric_nan": True,
        "preserve_categorical_missing": True,
        "imputation_applied": False,
        "encoding_applied": False,
        "scaling_applied": False,
        "binning_applied": False,
        "woe_applied": False,
        "outlier_cap_applied": False,
    },
    "numeric_strategy": {
        "logistic_regression": (
            "Model-specific bin or train-fitted imputation"
        ),
        "lightgbm": (
            "Native missing-value handling"
        ),
    },
    "categorical_strategy": {
        "missing": "Explicit reserved group",
        "rare": "Train-frequency rule",
        "unknown": "Reserved unknown group",
    },
}

missing_strategy_metadata_output = (
    paths.metadata
    / "application_missing_strategy.json"
)

with open(
    missing_strategy_metadata_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        missing_strategy_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Metadados de missing salvos em:",
    missing_strategy_metadata_output,
)

Metadados de missing salvos em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_missing_strategy.json


## Manifesto dos Dados Processados

O manifesto registra informações técnicas sobre os arquivos gerados, incluindo:

- nome e finalidade do arquivo;
- quantidade de registros e colunas;
- tamanho em disco;
- proporção de eventos;
- hash SHA-256;
- data e hora de geração.

O hash permite verificar se um arquivo foi alterado entre duas execuções. Arquivos com conteúdo binário diferente produzem hashes diferentes, mesmo que possuam o mesmo nome.

O hash valida a integridade do artefato persistido, mas pode variar caso o arquivo seja recriado com versões diferentes das bibliotecas de serialização, mesmo que os dados sejam semanticamente equivalentes.

In [114]:
# Função para cálculo do hash SHA-256
import hashlib
from datetime import datetime, timezone


def calculate_file_sha256(
    file_path,
    chunk_size=1024 * 1024,
):
    """
    Calcula o hash SHA-256 de um arquivo
    por meio de leitura em blocos.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [115]:
# Definição dos artefatos do manifesto
manifest_artifacts = [
    {
        "artifact_name": (
            "application_train_raw"
        ),
        "artifact_type": "raw_input",
        "file_path": (
            APPLICATION_TRAIN_PATH
        ),
        "dataframe": application_train,
    },
    {
        "artifact_name": (
            "application_split_assignment"
        ),
        "artifact_type": "interim",
        "file_path": (
            split_assignment_output
        ),
        "dataframe": split_assignment,
    },
    {
        "artifact_name": (
            "application_train_prepared"
        ),
        "artifact_type": "processed",
        "file_path": (
            processed_output_paths[
                "train"
            ]
        ),
        "dataframe": train_modeling,
    },
    {
        "artifact_name": (
            "application_validation_prepared"
        ),
        "artifact_type": "processed",
        "file_path": (
            processed_output_paths[
                "validation"
            ]
        ),
        "dataframe": validation_modeling,
    },
    {
        "artifact_name": (
            "application_test_prepared"
        ),
        "artifact_type": "processed",
        "file_path": (
            processed_output_paths[
                "test"
            ]
        ),
        "dataframe": test_modeling,
    },
]

In [116]:
# Construção do manifesto dos arquivos
manifest_generated_at = (
    datetime.now(timezone.utc)
    .isoformat()
)

data_manifest_records = []

for artifact in manifest_artifacts:
    file_path = Path(
        artifact["file_path"]
    )

    dataframe = artifact["dataframe"]

    if not file_path.exists():
        raise FileNotFoundError(
            "Artefato não encontrado: "
            f"{file_path}"
        )

    has_target = (
        TARGET_COLUMN
        in dataframe.columns
    )

    event_rate = (
        dataframe[TARGET_COLUMN].mean()
        if has_target
        else np.nan
    )

    data_manifest_records.append(
        {
            "artifact_name": (
                artifact["artifact_name"]
            ),
            "artifact_type": (
                artifact["artifact_type"]
            ),
            "file_name": file_path.name,
            "file_path": str(file_path),
            "n_records": len(dataframe),
            "n_columns": dataframe.shape[1],
            "event_rate": event_rate,
            "file_size_bytes": (
                file_path.stat().st_size
            ),
            "file_size_mb": (
                file_path.stat().st_size
                / 1024**2
            ),
            "sha256": (
                calculate_file_sha256(
                    file_path
                )
            ),
            "generated_at_utc": (
                manifest_generated_at
            ),
        }
    )

data_manifest = pd.DataFrame(
    data_manifest_records
)

display(
    data_manifest.style.format(
        {
            "n_records": "{:,.0f}",
            "n_columns": "{:,.0f}",
            "event_rate": "{:.2%}",
            "file_size_bytes": "{:,.0f}",
            "file_size_mb": "{:,.2f}",
        }
    )
)

,artifact_name,artifact_type,file_name,file_path,n_records,n_columns,event_rate,file_size_bytes,file_size_mb,sha256,generated_at_utc
0,application_train_raw,raw_input,application_train.csv,/content/drive/MyDrive/home-credit-default-risk/data/raw/application_train.csv,"307,511",122,8.07%,"166,133,370",158.44,52e96b895b1112e1c853f670e58372719c8441c5ed1c57ac2f7fad559d784f5f,2026-09-30T11:17:01.513102+00:00
1,application_split_assignment,interim,application_split_assignment.parquet,/content/drive/MyDrive/home-credit-default-risk/data/interim/application_split_assignment.parquet,"307,511",3,8.07%,"1,622,505",1.55,64503e91cc5027336601ff081b7d8e51189b1b137c3dde3fe887696cb31b61d9,2026-09-30T11:17:01.513102+00:00
2,application_train_prepared,processed,application_train_prepared.parquet,/content/drive/MyDrive/home-credit-default-risk/data/processed/application_train_prepared.parquet,"215,257",136,8.07%,"27,548,074",26.27,92476cc3f38f6e8a577deccb52a4b7155b5c704a42268a327a89100cb72727ea,2026-09-30T11:17:01.513102+00:00
3,application_validation_prepared,processed,application_validation_prepared.parquet,/content/drive/MyDrive/home-credit-default-risk/data/processed/application_validation_prepared.parquet,"46,127",136,8.07%,"6,821,894",6.51,8da452ec5415b911b0628bf035d9d1619066903930137a5b687ec977a6073bf9,2026-09-30T11:17:01.513102+00:00
4,application_test_prepared,processed,application_holdout_test_prepared.parquet,/content/drive/MyDrive/home-credit-default-risk/data/processed/application_holdout_test_prepared.parquet,"46,127",136,8.07%,"6,835,834",6.52,6f2e9a7cd95f98b2eb2c6c2bf36008a83f021d9f02da7b732614540790d9f4fd,2026-09-30T11:17:01.513102+00:00


In [117]:
# Validação do manifesto
manifest_validation = pd.DataFrame(
    {
        "validation": [
            "Todos os hashes foram gerados",
            "Não existem artefatos duplicados",
            "Todos os arquivos possuem registros",
            "Todos os arquivos possuem colunas",
            (
                "As bases processadas possuem "
                "o mesmo número de colunas"
            ),
        ],
        "result": [
            data_manifest[
                "sha256"
            ].str.len().eq(64).all(),

            not data_manifest[
                "artifact_name"
            ].duplicated().any(),

            data_manifest[
                "n_records"
            ].gt(0).all(),

            data_manifest[
                "n_columns"
            ].gt(0).all(),

            data_manifest.loc[
                data_manifest[
                    "artifact_type"
                ].eq("processed"),
                "n_columns",
            ].nunique() == 1,
        ],
    }
)

display(manifest_validation)

,validation,result
0,Todos os hashes foram gerados,True
1,Não existem artefatos duplicados,True
2,Todos os arquivos possuem registros,True
3,Todos os arquivos possuem colunas,True
4,As bases processadas possuem o mesmo número de...,True


In [118]:
# Interrupção se o manifesto for inválido
if not manifest_validation[
    "result"
].all():
    failed_validations = (
        manifest_validation.loc[
            ~manifest_validation["result"],
            "validation",
        ]
        .tolist()
    )

    raise ValueError(
        "Falhas na validação do manifesto: "
        f"{failed_validations}"
    )

print(
    "Manifesto validado com sucesso."
)

Manifesto validado com sucesso.


In [119]:
# Persistência do manifesto em CSV
data_manifest_output = (
    paths.metadata
    / "application_data_manifest.csv"
)

data_manifest.to_csv(
    data_manifest_output,
    index=False,
)

print(
    "Manifesto salvo em:",
    data_manifest_output,
)

Manifesto salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_data_manifest.csv


In [120]:
# Persistência do manifesto em JSON
data_manifest_json_output = (
    paths.metadata
    / "application_data_manifest.json"
)

with open(
    data_manifest_json_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "generated_at_utc": (
                manifest_generated_at
            ),
            "hash_algorithm": "SHA-256",
            "artifacts": (
                data_manifest
                .replace({np.nan: None})
                .to_dict(
                    orient="records"
                )
            ),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Manifesto JSON salvo em:",
    data_manifest_json_output,
)

Manifesto JSON salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_data_manifest.json


In [122]:
# Conferência dos arquivos processados
for sample_name, file_path in (
    processed_output_paths.items()
):
    print(
        f"{sample_name}: "
        f"{file_path.name} — "
        f"{file_path.stat().st_size / 1024**2:.2f} MB"
    )

train: application_train_prepared.parquet — 26.27 MB
validation: application_validation_prepared.parquet — 6.51 MB
test: application_holdout_test_prepared.parquet — 6.52 MB


## Consolidação dos Resultados

O notebook construiu uma base analítica comum a partir de `application_train.csv`, preservando a separação entre a preparação geral dos dados e os preprocessamentos específicos dos modelos.

As principais entregas foram:

- validação estrutural da tabela de aplicações;
- divisão estratificada em `train`, `validation` e holdout `test`;
- preservação da proporção do `TARGET`;
- garantia de ausência de clientes compartilhados entre as amostras;
- classificação das features por tipo;
- tratamento do código especial de `DAYS_EMPLOYED`;
- criação de features temporais, financeiras, familiares e de scores externos;
- validação da coerência das features criadas;
- remoção de identificadores, constantes, colunas totalmente ausentes e variáveis redundantes por construção;
- análise de categorias raras e desconhecidas;
- documentação da estratégia para valores ausentes;
- persistência das bases em Parquet;
- criação de manifesto com hashes SHA-256.

In [123]:
# Construção do resumo quantitativo
notebook_03_summary = pd.DataFrame(
    [
        {
            "indicator": "Registros na base original",
            "value": len(application_train),
        },
        {
            "indicator": "Registros em train",
            "value": len(train_modeling),
        },
        {
            "indicator": "Registros em validation",
            "value": len(validation_modeling),
        },
        {
            "indicator": (
                "Registros no holdout test"
            ),
            "value": len(test_modeling),
        },
        {
            "indicator": "Features originais",
            "value": len(feature_columns),
        },
        {
            "indicator": "Features criadas",
            "value": len(
                engineered_feature_columns
            ),
        },
        {
            "indicator": "Features excluídas",
            "value": len(
                excluded_feature_columns
            ),
        },
        {
            "indicator": (
                "Features candidatas finais"
            ),
            "value": len(
                model_feature_columns
            ),
        },
        {
            "indicator": (
                "Features numéricas finais"
            ),
            "value": len(
                numeric_model_features
            ),
        },
        {
            "indicator": (
                "Features categóricas finais"
            ),
            "value": len(
                categorical_model_features
            ),
        },
        {
            "indicator": (
                "Categorias raras em train"
            ),
            "value": len(
                rare_train_categories
            ),
        },
        {
            "indicator": (
                "Categorias desconhecidas"
            ),
            "value": len(
                unknown_categories
            ),
        },
    ]
)

display(
    notebook_03_summary.style.format(
        {
            "value": "{:,.0f}",
        }
    )
)

,indicator,value
0,Registros na base original,"307,511"
1,Registros em train,"215,257"
2,Registros em validation,"46,127"
3,Registros no holdout test,"46,127"
4,Features originais,120
5,Features criadas,19
6,Features excluídas,5
7,Features candidatas finais,134
8,Features numéricas finais,118
9,Features categóricas finais,16


In [124]:
# Definição das entregas obrigatórias
required_output_files = {
    "train": (
        paths.data_processed
        / "application_train_prepared.parquet"
    ),
    "validation": (
        paths.data_processed
        / "application_validation_prepared.parquet"
    ),
    "holdout_test": (
        paths.data_processed
        / "application_holdout_test_prepared.parquet"
    ),
    "split_assignment": (
        paths.data_interim
        / "application_split_assignment.parquet"
    ),
    "feature_schema": (
        paths.metadata
        / "application_feature_schema.csv"
    ),
    "engineered_features": (
        paths.metadata
        / "application_engineered_features.csv"
    ),
    "feature_exclusions": (
        paths.metadata
        / "application_feature_exclusions.csv"
    ),
    "model_features": (
        paths.metadata
        / "application_model_features.json"
    ),
    "data_manifest": (
        paths.metadata
        / "application_data_manifest.csv"
    ),
}

In [125]:
# Validação final das entregas
final_output_validation = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "file_path": str(file_path),
            "exists": file_path.exists(),
            "file_size_mb": (
                file_path.stat().st_size
                / 1024**2
                if file_path.exists()
                else np.nan
            ),
        }
        for artifact, file_path
        in required_output_files.items()
    ]
)

display(
    final_output_validation.style.format(
        {
            "file_size_mb": "{:,.2f}",
        }
    )
)

,artifact,file_path,exists,file_size_mb
0,train,/content/drive/MyDrive/home-credit-default-risk/data/processed/application_train_prepared.parquet,True,26.27
1,validation,/content/drive/MyDrive/home-credit-default-risk/data/processed/application_validation_prepared.parquet,True,6.51
2,holdout_test,/content/drive/MyDrive/home-credit-default-risk/data/processed/application_holdout_test_prepared.parquet,True,6.52
3,split_assignment,/content/drive/MyDrive/home-credit-default-risk/data/interim/application_split_assignment.parquet,True,1.55
4,feature_schema,/content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_feature_schema.csv,True,0.01
5,engineered_features,/content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_engineered_features.csv,True,0.00
6,feature_exclusions,/content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_feature_exclusions.csv,True,0.00
7,model_features,/content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_model_features.json,True,0.00
8,data_manifest,/content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/application_data_manifest.csv,True,0.00


In [126]:
# Interrupção se alguma entrega estiver ausente
if not final_output_validation[
    "exists"
].all():
    missing_artifacts = (
        final_output_validation.loc[
            ~final_output_validation["exists"],
            "artifact",
        ]
        .tolist()
    )

    raise FileNotFoundError(
        "Artefatos obrigatórios ausentes: "
        f"{missing_artifacts}"
    )

print(
    "Todas as entregas obrigatórias "
    "foram encontradas."
)

Todas as entregas obrigatórias foram encontradas.


In [127]:
# Persistência do resumo do notebook
notebook_03_summary_output = (
    paths.metadata
    / "notebook_03_summary.csv"
)

notebook_03_summary.to_csv(
    notebook_03_summary_output,
    index=False,
)

print(
    "Resumo salvo em:",
    notebook_03_summary_output,
)

Resumo salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/notebook_03_summary.csv


## Limitações

A preparação realizada possui algumas limitações importantes:

1. A divisão das amostras é aleatória e estratificada, não temporal.
2. O holdout test não representa uma amostra `Out-of-Time`.
3. A definição de `TARGET` é determinada pela base Home Credit e não corresponde necessariamente a uma definição regulatória de default.
4. As features produzidas neste notebook utilizam somente a tabela de aplicações.
5. As informações históricas das tabelas auxiliares ainda não foram agregadas.
6. Imputação, WoE, encoding, scaling, seleção de features e tratamento definitivo de categorias raras ainda não foram aplicados.
7. Os valores extremos foram preservados.
8. As features candidatas ainda precisarão ser avaliadas quanto a poder discriminante, estabilidade, redundância e interpretabilidade.

## Conclusão

O notebook produziu uma base de Application Score preparada de maneira reprodutível, com separação explícita entre `train`, `validation` e holdout `test`.

As transformações determinísticas foram aplicadas de forma consistente às três amostras, sem utilização do `TARGET` e sem aprendizado de parâmetros a partir de `validation` ou `test`.

A base processada preserva valores ausentes e categorias originais para que cada abordagem de modelagem utilize o tratamento mais adequado.

Neste momento, a base contém somente informações da aplicação atual. O próximo estágio será incorporar o histórico financeiro disponível nas tabelas auxiliares do Home Credit, criando agregações por cliente relacionadas a:

- registros de bureau;
- aplicações anteriores;
- parcelas e pagamentos;
- saldos de cartão de crédito;
- contratos de POS/CASH.

Somente depois da construção dessa visão consolidada serão desenvolvidos os pipelines dos modelos tradicional e challenger.

<!--  -->